# CS781 Phase 2 — legal NER, seven submission candidates, Kaggle T4 ×2

Reuse **your four Phase-1 foundations**. Run full/partial tuning, LoRA, **mandatory LoRMA+**, adapters and **smaller-student distillation**. Score exact character spans with all twelve official labels. This notebook contains all model, data, trainer and prediction source; it does not download external model weights or call an LLM.

**Before Run All:** attach your saved `capstone_phase1` directory with all four `runs/main_<architecture>/best_model.pt` files and the exact root `tokenizer.json`; select **GPU T4 ×2**. The five paths are configured for `/kaggle/input/datasets/hardik4tiwari/phase-1-outputs/capstone_phase1`. The loader reads only these files; archive extraction and searching other output trees are disabled. Keep the existing benchmarks, cache and submission folders as they are. Attach the six official Phase-2 files, or enable Internet and create an `HF_TOKEN` Kaggle secret after accepting dataset terms.

The original Drive capstone was updated October 9 and states **October 11, 2026, 11:59 PM** for Phase 2. Finish training, upload and allow scoring time before the deadline. Seven local ZIPs must still be uploaded individually; this notebook does not submit to the course platform.

Each finished candidate is saved immediately. The final cells download the seven ZIPs and archive the complete `/kaggle/working` directory. Real validation results decide which model works best; no private-test score is promised.

In [ ]:
import importlib.util, subprocess, sys
packages = {'tokenizers': 'tokenizers>=0.20,<0.24', 'huggingface_hub': 'huggingface_hub>=0.27,<3'}
missing = [spec for name, spec in packages.items() if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *missing], check=True)
import torch
if tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2]) < (2, 3):
    raise RuntimeError('Use a current Kaggle PyTorch image (2.3+); preserve its CUDA installation.')
print('PyTorch:', torch.__version__)

## Editable configuration

Defaults are a starting point, not a claim that these settings are optimal. Preflight measures real T4 memory and approximate step time. Two GPUs each hold a full model replica; effective batch is `2 × micro_batch × accumulation`.

**Time cap:** `max_stage_hours` applies separately to selection and optional all-label refit. Fourteen 0.30-hour stages allow approximately 4.2 hours inside training loops; end-of-stage validation, teacher caching, inference and ZIP creation add time. The cap is checked after an optimizer group, not enforced on the entire notebook. Set it and `refit_all` according to measured runtime and deadline. Check Kaggle's remaining session time.

For resume, keep configuration and GPU count unchanged. Rerun cells in the same working directory. To resume after a Kaggle session reset, extract the saved working archive back into `/kaggle/working` and reattach the original Phase-1 inputs. Changing training settings requires a fresh `PROJECT` directory. Interrupting a group resumes from the latest completed saved optimizer step.

In [ ]:
from pathlib import Path
import os, json, shutil, time
from IPython.display import display, FileLink
WORKING = Path('/kaggle/working')
PROJECT = WORKING / 'capstone_phase2'
CODE = PROJECT / 'code'
CODE.mkdir(parents=True, exist_ok=True)
INPUT_ROOT = Path('/kaggle/input')

# Your attached Kaggle Phase-1 dataset: exact paths supplied in this conversation.
# Directory-only loading: no recursive discovery and no input ZIP extraction.
PHASE1_ROOT = '/kaggle/input/datasets/hardik4tiwari/phase-1-outputs/capstone_phase1'
FOUNDATION_OVERRIDES = {
    'decoder': '/kaggle/input/datasets/hardik4tiwari/phase-1-outputs/capstone_phase1/runs/main_decoder/best_model.pt',
    'encoder_decoder': '/kaggle/input/datasets/hardik4tiwari/phase-1-outputs/capstone_phase1/runs/main_encoder_decoder/best_model.pt',
    'local_gqa': '/kaggle/input/datasets/hardik4tiwari/phase-1-outputs/capstone_phase1/runs/main_local_gqa/best_model.pt',
    'hybrid': '/kaggle/input/datasets/hardik4tiwari/phase-1-outputs/capstone_phase1/runs/main_hybrid/best_model.pt',
}
TOKENIZER_OVERRIDE = '/kaggle/input/datasets/hardik4tiwari/phase-1-outputs/capstone_phase1/tokenizer.json'
DATA_ROOT = None  # folder containing Train/ and Test/, or a parent of Phase_2_LNER/
SEQ_LEN = 512
OVERLAP = 128
VALIDATION_FRACTION = 0.20
SEED = 230437
SETTINGS = {
    'gpus': 2,
    'epochs': 12,
    'micro_batch': 4,
    'accumulation': 4,
    'max_stage_hours': 0.30,
    'refit_all': True,
    'student_layers': 4,
    'gradient_checkpointing': False,
    'memory_budget_gib': 12.5,
}
RUN_PREFLIGHT = True
if torch.cuda.device_count() != 2:
    raise RuntimeError('Select Kaggle GPU T4 ×2 before running this notebook.')
for i in range(2):
    gpu = torch.cuda.get_device_properties(i)
    print(i, gpu.name, round(gpu.total_memory / 1024**3, 2), 'GiB')
    if 'T4' not in gpu.name:
        print('Hardware differs from the requested T4; use measured preflight results.')
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['OMP_NUM_THREADS'] = '1'
sys.path.insert(0, str(CODE))

## Embedded implementation

The foundation module below is weight-compatible with the previous four-foundation Phase-1 notebook. Task-specific attention masks preserve each architecture's causality and add padding handling. Causal foundations share weights across forward/reversed windows to obtain both contexts; the encoder–decoder uses the same input text as encoder memory. No gold labels enter model inputs. LoRMA+ is multiplicative on the frozen projection's output; LoRA is additive on its input.

The next cells write executable modules under the project directory. Keep them with your checkpoints for reproducibility.

In [ ]:
MODULE_SOURCE = r'''
"""Four randomly initialized legal language models; no downloaded model weights."""
import math
from dataclasses import dataclass, asdict
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint


@dataclass
class ModelConfig:
    architecture: str = "decoder"
    vocab_size: int = 32768
    dim: int = 512
    heads: int = 8
    kv_heads: int = 8
    layers: int = 8
    ff_dim: int = 2048
    encoder_layers: int = 0
    rnn_layers: int = 0
    local_window: int = 0
    max_positions: int = 4096
    dropout: float = 0.0
    gradient_checkpointing: bool = False


def architecture_config(name, **overrides):
    c = ModelConfig(architecture=name)
    if name == "decoder":
        pass
    elif name == "local_gqa":
        c.kv_heads, c.layers, c.local_window = 2, 9, 256
    elif name == "hybrid":
        c.layers, c.rnn_layers = 7, 2
    elif name == "encoder_decoder":
        c.layers, c.encoder_layers = 4, 3
    else:
        raise ValueError(f"Unknown architecture: {name}")
    for k, v in overrides.items():
        if not hasattr(c, k):
            raise ValueError(k)
        setattr(c, k, v)
    assert c.dim % c.heads == 0 and c.heads % c.kv_heads == 0
    assert (c.dim // c.heads) % 2 == 0
    return c


class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        # Reduction in FP32 is useful with T4 FP16 training.
        y = x.float() * torch.rsqrt(x.float().pow(2).mean(-1, keepdim=True) + self.eps)
        return y.to(x.dtype) * self.weight.to(x.dtype)


class Rotary(nn.Module):
    def __init__(self, head_dim, max_positions):
        super().__init__()
        inv = 1.0 / (10000 ** (torch.arange(0, head_dim, 2).float() / head_dim))
        angles = torch.outer(torch.arange(max_positions).float(), inv)
        self.register_buffer("cos", angles.cos(), persistent=False)
        self.register_buffer("sin", angles.sin(), persistent=False)

    def forward(self, x, offset=0):
        n = x.size(-2)
        if offset + n > self.cos.size(0):
            raise ValueError("Position limit exceeded; increase max_positions in the model config.")
        cos = self.cos[offset:offset + n].to(x.dtype)[None, None]
        sin = self.sin[offset:offset + n].to(x.dtype)[None, None]
        a, b = x[..., 0::2], x[..., 1::2]
        return torch.stack((a * cos - b * sin, a * sin + b * cos), dim=-1).flatten(-2)


class Attention(nn.Module):
    def __init__(self, c, causal=True, cross=False):
        super().__init__()
        self.heads, self.kv_heads = c.heads, c.kv_heads
        self.hd = c.dim // c.heads
        self.q = nn.Linear(c.dim, c.dim, bias=False)
        self.k = nn.Linear(c.dim, self.kv_heads * self.hd, bias=False)
        self.v = nn.Linear(c.dim, self.kv_heads * self.hd, bias=False)
        self.out = nn.Linear(c.dim, c.dim, bias=False)
        self.rotary = None if cross else Rotary(self.hd, c.max_positions)
        self.causal, self.cross = causal, cross
        self.window = c.local_window if causal and not cross else 0
        self.dropout = c.dropout

    def forward(self, x, memory=None, past=None, position=0, use_cache=False):
        b, t, _ = x.shape
        q = self.q(x).view(b, t, self.heads, self.hd).transpose(1, 2)
        if self.cross and past is not None:
            k, v = past  # Encoder K/V projections are also cached during decoding.
        else:
            source = x if memory is None else memory
            s = source.size(1)
            k = self.k(source).view(b, s, self.kv_heads, self.hd).transpose(1, 2)
            v = self.v(source).view(b, s, self.kv_heads, self.hd).transpose(1, 2)
            if self.rotary is not None:
                k = self.rotary(k, position)
            if past is not None:
                k, v = torch.cat((past[0], k), -2), torch.cat((past[1], v), -2)
        if self.rotary is not None:
            q = self.rotary(q, position)
        new_cache = (k, v) if use_cache else None
        # Portable GQA: project/cache fewer KV heads, repeat only for the SDPA call.
        # This is not a block-sparse implementation or a guaranteed training speedup.
        if self.kv_heads != self.heads:
            repeats = self.heads // self.kv_heads
            k = k.repeat_interleave(repeats, dim=1)
            v = v.repeat_interleave(repeats, dim=1)
        mask, is_causal = None, False
        if self.causal:
            s = k.size(-2)
            if self.window or (past is not None and t > 1):
                qi = torch.arange(s - t, s, device=x.device)[:, None]
                ki = torch.arange(s, device=x.device)[None, :]
                mask = ki <= qi
                if self.window:
                    mask = mask & ((qi - ki) < self.window)
            elif past is None:
                is_causal = True
            # Cached single-token query must see all prior keys. is_causal=True
            # here would incorrectly use the upper-left non-square mask.
        y = F.scaled_dot_product_attention(
            q, k, v, attn_mask=mask, is_causal=is_causal,
            dropout_p=self.dropout if self.training else 0.0,
        )
        y = self.out(y.transpose(1, 2).contiguous().view(b, t, -1))
        return y, new_cache


class SwiGLU(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.gate = nn.Linear(c.dim, c.ff_dim, bias=False)
        self.up = nn.Linear(c.dim, c.ff_dim, bias=False)
        self.down = nn.Linear(c.ff_dim, c.dim, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))


class Block(nn.Module):
    def __init__(self, c, encoder=False):
        super().__init__()
        self.attn_norm, self.ff_norm = RMSNorm(c.dim), RMSNorm(c.dim)
        self.attn = Attention(c, causal=not encoder)
        self.ff = SwiGLU(c)
        self.drop = nn.Dropout(c.dropout)

    def forward(self, x, past=None, position=0, use_cache=False):
        y, cache = self.attn(self.attn_norm(x), past=past, position=position, use_cache=use_cache)
        x = x + self.drop(y)
        x = x + self.drop(self.ff(self.ff_norm(x)))
        return (x, cache) if use_cache else x


class CrossBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.self_norm, self.cross_norm, self.ff_norm = (RMSNorm(c.dim) for _ in range(3))
        self.self_attn = Attention(c)
        self.cross_attn = Attention(c, causal=False, cross=True)
        self.ff, self.drop = SwiGLU(c), nn.Dropout(c.dropout)

    def forward(self, x, memory, past=None, cross_past=None, position=0, use_cache=False):
        y, cache = self.self_attn(self.self_norm(x), past=past, position=position, use_cache=use_cache)
        x = x + self.drop(y)
        y, cross_cache = self.cross_attn(self.cross_norm(x), memory=memory, past=cross_past, use_cache=use_cache)
        x = x + self.drop(y)
        x = x + self.drop(self.ff(self.ff_norm(x)))
        return (x, cache, cross_cache) if use_cache else x


class LegalLM(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.config = c
        self.embedding = nn.Embedding(c.vocab_size, c.dim)
        # Output projection uses embedding.weight; there is no second vocabulary matrix.
        if c.architecture == "encoder_decoder":
            self.encoder = nn.ModuleList(Block(c, encoder=True) for _ in range(c.encoder_layers))
            self.encoder_norm = RMSNorm(c.dim)
            self.blocks = nn.ModuleList(CrossBlock(c) for _ in range(c.layers))
        else:
            self.blocks = nn.ModuleList(Block(c) for _ in range(c.layers))
        if c.rnn_layers:
            self.rnn = nn.LSTM(c.dim, c.dim, num_layers=c.rnn_layers, batch_first=True)
            self.rnn_norm = RMSNorm(c.dim)
        self.final_norm = RMSNorm(c.dim)
        self.apply(self._init)
        # Scale residual projections to keep initial residual variance controlled.
        std = 0.02 / math.sqrt(2 * c.layers)
        for name, p in self.named_parameters():
            if name.endswith(("attn.out.weight", "self_attn.out.weight", "cross_attn.out.weight", "ff.down.weight")):
                nn.init.normal_(p, mean=0.0, std=std)

    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
        elif isinstance(m, nn.LSTM):
            for name, p in m.named_parameters():
                if "weight" in name:
                    nn.init.normal_(p, mean=0.0, std=0.02)
                else:
                    nn.init.zeros_(p)

    def encode(self, source_ids):
        x = self.embedding(source_ids)
        for block in self.encoder:
            if self.config.gradient_checkpointing and self.training:
                x = checkpoint(block, x, use_reentrant=False)
            else:
                x = block(x)
        return self.encoder_norm(x)

    def forward(self, input_ids, source_ids=None, state=None, use_cache=False):
        state = state or {}
        position = state.get("position", 0)
        x = self.embedding(input_ids)
        new_state = {"position": position + input_ids.size(1), "self": []}
        if self.config.rnn_layers:
            y, recurrent_state = self.rnn(x, state.get("rnn"))
            x = x + self.rnn_norm(y)
            new_state["rnn"] = recurrent_state
        if self.config.architecture == "encoder_decoder":
            memory = state.get("memory")
            if memory is None:
                if source_ids is None:
                    raise ValueError("encoder_decoder requires source_ids on the first call.")
                memory = self.encode(source_ids)
            new_state.update(memory=memory, cross=[])
            for i, block in enumerate(self.blocks):
                if use_cache:
                    p = state.get("self", [None] * len(self.blocks))[i]
                    cp = state.get("cross", [None] * len(self.blocks))[i]
                    x, cache, cc = block(x, memory, p, cp, position, True)
                    new_state["self"].append(cache)
                    new_state["cross"].append(cc)
                elif self.config.gradient_checkpointing and self.training:
                    x = checkpoint(block, x, memory, use_reentrant=False)
                else:
                    x = block(x, memory)
        else:
            for i, block in enumerate(self.blocks):
                if use_cache:
                    p = state.get("self", [None] * len(self.blocks))[i]
                    x, cache = block(x, p, position, True)
                    new_state["self"].append(cache)
                elif self.config.gradient_checkpointing and self.training:
                    x = checkpoint(block, x, use_reentrant=False)
                else:
                    x = block(x)
        x = self.final_norm(x)
        return (x, new_state) if use_cache else x

    def project(self, hidden):
        return F.linear(hidden, self.embedding.weight)


def parameter_count(model):
    # PyTorch de-duplicates shared Parameter objects here.
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


def parameter_formula(c):
    d, h, kv = c.dim, c.dim // c.heads, c.kv_heads
    attention = 2 * d * d + 2 * d * kv * h
    ff = 3 * d * c.ff_dim
    total = c.vocab_size * d + d
    if c.architecture == "encoder_decoder":
        total += c.encoder_layers * (attention + ff + 2 * d) + d
        total += c.layers * (2 * attention + ff + 3 * d)
    else:
        total += c.layers * (attention + ff + 2 * d)
    if c.rnn_layers:
        total += c.rnn_layers * (8 * d * d + 8 * d) + d
    return total


def make_batch(blocks, architecture, prefix_fraction=0.80, objective="full"):
    # blocks has T+1 tokens. The target suffix is identical across architectures.
    t = blocks.size(1) - 1
    split = max(1, min(t, int(t * prefix_fraction)))
    if architecture == "encoder_decoder":
        source = blocks[:, :split]
        inputs, targets = blocks[:, split - 1:-1], blocks[:, split:]
    else:
        source = None
        inputs, targets = blocks[:, :-1], blocks[:, 1:].clone()
        if objective == "continuation":
            targets[:, :split - 1] = -100
    return inputs, targets, source


def language_model_loss(hidden, targets, weight, chunk_tokens=256, checkpoint_chunks=False):
    """Sum/token count lets validation aggregate the true token-weighted NLL."""
    hidden, targets = hidden.reshape(-1, hidden.size(-1)), targets.reshape(-1)
    total = hidden.new_zeros((), dtype=torch.float32)
    count = (targets != -100).sum()
    def loss_chunk(h, y, w):
        # Keep logits in autocast precision until the bounded FP32 CE conversion.
        return F.cross_entropy(F.linear(h, w).float(), y, ignore_index=-100, reduction="sum")
    for i in range(0, targets.numel(), chunk_tokens):
        h, y = hidden[i:i + chunk_tokens], targets[i:i + chunk_tokens]
        if checkpoint_chunks and torch.is_grad_enabled():
            total = total + checkpoint(loss_chunk, h, y, weight, use_reentrant=False)
        else:
            total = total + loss_chunk(h, y, weight)
    return total / count.clamp_min(1), total.detach(), count.detach()


if __name__ == "__main__":
    for name in ("decoder", "encoder_decoder", "local_gqa", "hybrid"):
        c = architecture_config(name)
        with torch.device("meta"):
            model = LegalLM(c)
        n = parameter_count(model)
        assert n == parameter_formula(c) and 50_000_000 <= n <= 300_000_000
        print(name, f"{n:,}", asdict(c))
'''
(CODE / 'legal_models.py').write_text(MODULE_SOURCE.lstrip('\n'), encoding='utf-8')
print('Wrote', 'legal_models.py')

In [ ]:
MODULE_SOURCE = r'''
"""Official L-NER data, unchanged text, document split, and character alignment."""
import hashlib
import json
import math
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import Dataset
from tokenizers import Tokenizer

OFFICIAL_LABELS = {'APP','RESP','A.COUNSEL','R.COUNSEL','JUDGE','WIT','AUTH','COURT','STAT','PREC','DATE','CASENO'}

def digest(obj):
    return hashlib.sha256(json.dumps(obj,sort_keys=True,ensure_ascii=False,separators=(',',':')).encode()).hexdigest()

def file_hash(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for b in iter(lambda:f.read(1<<20),b''): h.update(b)
    return h.hexdigest()

def atomic_json(path,data):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    tmp=path.with_name(path.name+'.tmp')
    tmp.write_text(json.dumps(data,ensure_ascii=False,indent=2),encoding='utf-8');tmp.replace(path)

def no_duplicates(pairs):
    data={}
    for key,value in pairs:
        if key in data: raise ValueError(f'Duplicate JSON key: {key}')
        data[key]=value
    return data

def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8-sig'),object_pairs_hook=no_duplicates)

def validate_spans(predictions,texts,labels):
    if not isinstance(predictions,dict) or set(predictions)!=set(texts): raise ValueError('Prediction IDs must match document IDs exactly.')
    for did,entities in predictions.items():
        if not isinstance(entities,list): raise ValueError(f'{did}: entities must be a list.')
        seen=set()
        for span in entities:
            if not isinstance(span,list) or len(span) not in (3,4): raise ValueError(f'{did}: invalid span schema.')
            start,end,label=span[:3]
            if type(start) is not int or type(end) is not int or not 0<=start<end<=len(texts[did]):
                raise ValueError(f'{did}: invalid character offsets {span[:2]}.')
            if label not in labels: raise ValueError(f'{did}: unknown label {label}.')
            if len(span)==4 and span[3]!=texts[did][start:end]: raise ValueError(f'{did}: entity text differs from its exact source slice.')
            key=tuple(span[:3])
            if key in seen: raise ValueError(f'{did}: repeated entity.')
            seen.add(key)
    return True

def load_official(root):
    root=Path(root)
    if (root/'Phase_2_LNER').is_dir(): root=root/'Phase_2_LNER'
    labels=[x.strip() for x in (root/'Train/labels.txt').read_text(encoding='utf-8-sig').splitlines() if x.strip()]
    if len(labels)!=12 or set(labels)!=OFFICIAL_LABELS: raise ValueError('Official 12-label vocabulary mismatch.')
    texts,gold,folds={},{},{}
    for fold in (1,2):
        docs=read_json(root/f'Train/fold{fold}-text.json');annotations=read_json(root/f'Train/fold{fold}-gold.json')
        if set(docs)!=set(annotations): raise ValueError(f'Fold {fold}: document/gold ID mismatch.')
        if set(texts)&set(docs): raise ValueError('Repeated IDs across training folds.')
        if not all(isinstance(k,str) and isinstance(v,str) for k,v in docs.items()): raise ValueError('Expected ID -> original document string.')
        # The official evaluator ignores the optional fourth field. Its wording may
        # differ from the document slice; character offsets and label remain authoritative.
        for key in annotations:
            if annotations[key] is None:annotations[key]=[]
        validate_spans({k:[e[:3] for e in v] for k,v in annotations.items()},docs,labels)
        texts.update(docs);gold.update(annotations);folds.update({key:fold for key in docs})
    test=read_json(root/'Test/fold3-text.json')
    if not test or not all(isinstance(k,str) and isinstance(v,str) for k,v in test.items()): raise ValueError('Official test document schema mismatch.')
    if set(texts)&set(test): raise ValueError('Train/test ID overlap.')
    return texts,gold,test,labels,folds

def grouped_split(texts,gold,labels,fraction=.2,seed=230437):
    groups={}
    for did,text in texts.items():
        # Group whitespace/case duplicates for split hygiene without altering model input.
        key=hashlib.sha256(' '.join(text.split()).casefold().encode()).hexdigest()
        groups.setdefault(key,[]).append(did)
    keys=sorted(groups,key=lambda k:digest([seed,k]))
    if len(keys)<2 or not 0<fraction<1:raise ValueError('Need two distinct groups and a validation fraction between zero and one.')
    n=min(len(keys)-1,max(1,round(len(keys)*fraction)))
    presence={k:np.array([any(e[2]==label for did in groups[k] for e in gold[did]) for label in labels],dtype=np.int64) for k in keys}
    total=sum(presence.values());target=np.maximum(1,np.round(total*fraction));target[total<2]=0
    chosen=[];counts=np.zeros(len(labels),dtype=np.int64)
    for _ in range(n):
        options=[k for k in keys if k not in chosen and np.all((total-counts-presence[k]>=1)|(total==0))]
        if not options:break
        def value(k):
            before=np.sum(np.abs(target-counts)/np.maximum(total,1))
            after=np.sum(np.abs(target-counts-presence[k])/np.maximum(total,1))
            return before-after
        best=max(options,key=value);chosen.append(best);counts+=presence[best]
    if not chosen:
        # If every group owns a singleton label, a nonempty holdout necessarily
        # removes some training support. Choose the least damaging group and report it.
        best=min(keys,key=lambda k:int(presence[k][total==1].sum()));chosen=[best];counts=presence[best].copy()
    val_ids=sorted(did for k in chosen for did in groups[k]);train_ids=sorted(set(texts)-set(val_ids))
    if not train_ids or not val_ids: raise ValueError('Need at least two distinct document groups.')
    report={'seed':seed,'validation_fraction_requested':fraction,'train_ids':train_ids,'val_ids':val_ids,
        'duplicate_groups':len(texts)-len(groups),'validation_groups_by_label':dict(zip(labels,map(int,counts))),
        'training_groups_by_label':dict(zip(labels,map(int,total-counts))),
        'actual_validation_group_fraction':len(chosen)/len(keys),
        'training_absent_labels':[l for l,c in zip(labels,total-counts) if c==0],
        'validation_absent_labels':[l for l,c in zip(labels,counts) if c==0]}
    return train_ids,val_ids,report

def encode_documents(texts,tokenizer_path):
    tokenizer=Tokenizer.from_file(str(tokenizer_path));tokenizer.no_truncation();tokenizer.no_padding()
    encoded={}
    for did,text in texts.items():
        result=tokenizer.encode(text,add_special_tokens=False)
        ids,offsets=[],[]
        for tid,(a,b) in zip(result.ids,result.offsets):
            if a==b: continue
            if not 0<=a<b<=len(text): raise ValueError(f'{did}: tokenizer offsets are not Python character offsets.')
            ids.append(tid);offsets.append((a,b))
        encoded[did]={'ids':ids,'offsets':offsets}
    return encoded

def align_entities(encoded,gold,labels):
    aligned={};audit={'entities':0,'max_entity_tokens':0,'max_token_chars':0,'bio_dropped_overlaps':0,'uncovered':[]}
    for did,doc in encoded.items():
        offsets=doc['offsets'];n=len(offsets)
        audit['max_token_chars']=max(audit['max_token_chars'],max((b-a for a,b in offsets),default=1))
        starts=np.array([a for a,b in offsets]);ends=np.array([b for a,b in offsets])
        entities=[]
        for span in gold.get(did,[]):
            a,b,label=span[:3];hits=np.flatnonzero((ends>a)&(starts<b))
            if not len(hits): audit['uncovered'].append([did,*span[:3]]);continue
            i,j=int(hits[0]),int(hits[-1])
            if not offsets[i][0]<=a<offsets[i][1] or not offsets[j][0]<b<=offsets[j][1]:
                raise ValueError(f'{did}: gold boundary cannot be represented by tokenizer offsets.')
            entities.append((i,j,labels.index(label),a-offsets[i][0],offsets[j][1]-b))
            audit['entities']+=1;audit['max_entity_tokens']=max(audit['max_entity_tokens'],j-i+1)
        tags=np.zeros(n,dtype=np.int64);occupied=np.zeros(n,dtype=bool)
        kept=[]
        for i,j,l,left,right in sorted(entities,key=lambda x:(-(x[1]-x[0]),x[0],x[2])):
            if occupied[i:j+1].any():audit['bio_dropped_overlaps']+=1;continue
            tags[i]=1+2*l;tags[i+1:j+1]=2+2*l;occupied[i:j+1]=True;kept.append((i,j,l,left,right))
        aligned[did]={'entities':entities,'bio_entities':kept,'tags':tags}
    if audit['uncovered']: raise ValueError(f'Uncovered annotations: {audit["uncovered"][:3]}')
    return aligned,audit

def windows_for(encoded,ids,seq_len,overlap):
    if not 0<=overlap<seq_len:raise ValueError('Invalid context overlap.')
    windows=[]
    for did in ids:
        n=len(encoded[did]['ids']);start=0
        if not n:continue
        while True:
            end=min(start+seq_len,n);windows.append({'did':did,'start':start,'end':end})
            if end==n:break
            start=end-overlap
    return windows

def create_cache(texts,gold,test,labels,tokenizer_path,path,seq_len=512,overlap=128,seed=230437,val_fraction=.2):
    path=Path(path);source_id=digest({'texts':texts,'gold':gold,'test':test,'labels':labels,'tokenizer':file_hash(tokenizer_path),
                                   'seq_len':seq_len,'overlap':overlap,'seed':seed,'val_fraction':val_fraction,'version':3})
    if path.exists():
        old=torch.load(path,map_location='cpu',weights_only=False)
        if old['fingerprint']==source_id:return old
        raise ValueError('Token cache configuration changed; choose a new cache filename.')
    encoded=encode_documents({**texts,**test},tokenizer_path);aligned,audit=align_entities(encoded,gold,labels)
    if audit['max_entity_tokens']>seq_len:raise ValueError(f'Increase SEQ_LEN: longest gold entity uses {audit["max_entity_tokens"]} tokens.')
    effective_overlap=max(overlap,audit['max_entity_tokens']-1)
    if effective_overlap>=seq_len:raise ValueError('Context window too short for annotated entities.')
    char_classes=max(8,audit['max_token_chars']+1)
    if char_classes>256:raise ValueError('Tokenizer has unusually long tokens; inspect before allocating boundary heads.')
    train_ids,val_ids,split=grouped_split(texts,gold,labels,val_fraction,seed)
    cache={'fingerprint':source_id,'texts':{**texts,**test},'gold':gold,'labels':labels,'encoded':encoded,'aligned':aligned,
        'split':split,'audit':audit,'seq_len':seq_len,'overlap':effective_overlap,'char_classes':char_classes,
        'tokenizer_sha256':file_hash(tokenizer_path)}
    for name,doc_ids in [('train',train_ids),('val',val_ids),('all',sorted(texts)),('test',list(test))]:
        cache[name+'_windows']=windows_for(encoded,doc_ids,seq_len,effective_overlap)
    # Every labelled entity must fit in at least one inference-compatible training window.
    for did,alignment in aligned.items():
        if did not in gold:continue
        local=[w for w in cache['all_windows'] if w['did']==did]
        for i,j,*_ in alignment['entities']:
            if not any(w['start']<=i<=j<w['end'] for w in local):raise ValueError('A gold span falls between windows.')
    audit['gold_entity_text_mismatches']=sum(len(e)==4 and e[3]!=texts[k][e[0]:e[1]] for k,entities in gold.items() for e in entities)
    path.parent.mkdir(parents=True,exist_ok=True);temp=path.with_suffix('.tmp.pt');torch.save(cache,temp);temp.replace(path)
    return cache

class NERWindows(Dataset):
    def __init__(self,cache,split,head,teacher_dir=None):
        self.cache,self.windows,self.head=cache,cache[split+'_windows'],head
        self.teacher_dir=Path(teacher_dir) if teacher_dir else None
    def __len__(self):return len(self.windows)
    def __getitem__(self,index):
        window=self.windows[index];did,start,end=window['did'],window['start'],window['end']
        n=end-start;L=len(self.cache['labels']);doc=self.cache['encoded'][did]
        item={'ids':torch.tensor(doc['ids'][start:end]),'offsets':doc['offsets'][start:end],
              'index':index,'window':window,'tags':torch.zeros(n,dtype=torch.long),
              'left':torch.full((n,L),-100,dtype=torch.long),'right':torch.full((n,L),-100,dtype=torch.long),'entities':[]}
        alignment=self.cache['aligned'][did]
        if self.head=='bio':item['tags']=torch.from_numpy(alignment['tags'][start:end].copy())
        use=alignment['bio_entities'] if self.head=='bio' else alignment['entities']
        for i,j,label,left,right in use:
            if start<=i<=j<end:
                item['entities'].append((i-start,j-start,label))
                item['left'][i-start,label]=left;item['right'][j-start,label]=right
        if self.teacher_dir:
            item['teacher']=torch.from_numpy(np.load(self.teacher_dir/f'{index:06d}.npz')['logits'].astype(np.float32))
        return item

def collate_windows(items,pad_id=0):
    size=max(len(x['ids']) for x in items);batch=len(items);L=items[0]['left'].size(1)
    ids=torch.full((batch,size),pad_id,dtype=torch.long);valid=torch.zeros(batch,size,dtype=torch.bool)
    tags=torch.full((batch,size),-100,dtype=torch.long);left=torch.full((batch,size,L),-100,dtype=torch.long);right=left.clone()
    teacher=None
    if 'teacher' in items[0]:teacher=torch.zeros(batch,size,items[0]['teacher'].size(-1))
    for row,item in enumerate(items):
        n=len(item['ids']);ids[row,:n]=item['ids'];valid[row,:n]=True;tags[row,:n]=item['tags']
        left[row,:n]=item['left'];right[row,:n]=item['right']
        if teacher is not None:teacher[row,:n]=item['teacher']
    return {'ids':ids,'valid':valid,'tags':tags,'left':left,'right':right,'teacher':teacher,'items':items}

def class_weights(cache,split):
    counts=np.ones(1+2*len(cache['labels']),dtype=np.float64)
    for did in cache['split']['train_ids'] if split=='train' else cache['gold']:
        counts+=np.bincount(cache['aligned'][did]['tags'],minlength=len(counts))
    weights=np.sqrt(counts.sum()/counts);weights/=weights[1:].mean();weights=np.clip(weights,.15,5.)
    return torch.tensor(weights,dtype=torch.float32)

def strict_metrics(predictions,gold,labels):
    if set(predictions)!=set(gold):raise ValueError('Metric ID mismatch.')
    rows=[]
    for label in labels:
        tp=fp=fn=0
        for did in gold:
            p={tuple(x[:3]) for x in predictions[did] if x[2]==label};g={tuple(x[:3]) for x in gold[did] if x[2]==label}
            tp+=len(p&g);fp+=len(p-g);fn+=len(g-p)
        precision=tp/(tp+fp) if tp+fp else 0;recall=tp/(tp+fn) if tp+fn else 0
        f1=2*tp/(2*tp+fp+fn) if 2*tp+fp+fn else 0
        rows.append({'label':label,'tp':tp,'fp':fp,'fn':fn,'precision':precision,'recall':recall,'f1':f1})
    return {'macro_f1':sum(x['f1'] for x in rows)/len(labels),'macro_precision':sum(x['precision'] for x in rows)/len(labels),
            'macro_recall':sum(x['recall'] for x in rows)/len(labels),'per_label':rows}
'''
(CODE / 'ner_data.py').write_text(MODULE_SOURCE.lstrip('\n'), encoding='utf-8')
print('Wrote', 'ner_data.py')

In [ ]:
MODULE_SOURCE = r'''
"""Own Phase-1 weights adapted by full/partial tuning, LoRA, LoRMA+, adapters, KD."""
import math
import types
from dataclasses import asdict
import torch
from torch import nn
from torch.nn import functional as F
from legal_models import LegalLM,ModelConfig,Attention,RMSNorm,Rotary
from ner_data import file_hash

class LowRankLinear(nn.Module):
    def __init__(self,base,rank=8,alpha=16.,multiplicative=False):
        super().__init__();self.base=base;self.scale=alpha/rank;self.multiplicative=multiplicative
        self.A=nn.Linear(base.out_features if multiplicative else base.in_features,rank,bias=False)
        self.B=nn.Linear(rank,base.out_features,bias=False)
        nn.init.normal_(self.A.weight,std=.02);nn.init.zeros_(self.B.weight)
        self.base.requires_grad_(False)
    def forward(self,x):
        y=self.base(x)
        # LoRMA+ Eq. 8: (I + alpha/r BA) W0 x. Projection targets are bias-free.
        return y+self.scale*self.B(self.A(y if self.multiplicative else x))
    def merged_weight(self):
        change=self.B.weight@self.A.weight
        return self.base.weight+self.scale*(change@self.base.weight if self.multiplicative else change)

class AdapterBlock(nn.Module):
    """Post-block residual bottleneck variant; not an exact Houlsby reproduction."""
    def __init__(self,base,dim,bottleneck):
        super().__init__();self.base=base;self.norm=RMSNorm(dim)
        self.down=nn.Linear(dim,bottleneck);self.up=nn.Linear(bottleneck,dim)
        nn.init.zeros_(self.up.weight);nn.init.zeros_(self.up.bias)
    def forward(self,*args,**kwargs):
        x=self.base(*args,**kwargs)
        if isinstance(x,tuple):raise ValueError('NER adapters use full-sequence inference.')
        return x+self.up(F.gelu(self.down(self.norm(x))))

def ner_attention(self,x,memory=None,past=None,position=0,use_cache=False):
    if past is not None or use_cache:raise ValueError('NER attention is full-sequence, not LM cached generation.')
    b,t,_=x.shape;source=x if memory is None else memory;s=source.size(1)
    q=self.q(x).view(b,t,self.heads,self.hd).transpose(1,2)
    k=self.k(source).view(b,s,self.kv_heads,self.hd).transpose(1,2)
    v=self.v(source).view(b,s,self.kv_heads,self.hd).transpose(1,2)
    if self.rotary is not None:q=self.rotary(q);k=self.rotary(k)
    if self.kv_heads!=self.heads:
        k=k.repeat_interleave(self.heads//self.kv_heads,1);v=v.repeat_interleave(self.heads//self.kv_heads,1)
    valid=getattr(self,'_ner_valid',None);mask=None;causal=self.causal
    if self.window or valid is not None:
        mask=torch.ones(t,s,device=x.device,dtype=torch.bool)
        if self.causal:
            qi=torch.arange(t,device=x.device)[:,None];ki=torch.arange(s,device=x.device)[None,:]
            mask=ki<=qi
            if self.window:mask&=(qi-ki)<self.window
        if valid is not None:mask=mask[None,None]&valid[:,None,None,:s]
        causal=False
    y=F.scaled_dot_product_attention(q,k,v,attn_mask=mask,is_causal=causal,
        dropout_p=self.dropout if self.training else 0.)
    return self.out(y.transpose(1,2).contiguous().view(b,t,-1)),None

def load_foundation(path,tokenizer_hash,architecture,student_layers=None,allow_tiny=False):
    saved=torch.load(path,map_location='cpu',weights_only=False)
    if saved.get('training_origin')!='random_initialization_official_phase1_only':raise ValueError('Use your own Phase-1 checkpoint.')
    if saved['tokenizer_sha256']!=tokenizer_hash:raise ValueError('Checkpoint/tokenizer mismatch.')
    c=ModelConfig(**saved['model_config'])
    if c.architecture!=architecture:raise ValueError('Foundation architecture mismatch.')
    parent_layers=c.layers;source=saved['model'];retained=None
    if student_layers is not None:
        if architecture!='decoder' or not 1<=student_layers<parent_layers:raise ValueError('Student must retain fewer decoder blocks.')
        retained=[round(i*(parent_layers-1)/(student_layers-1)) for i in range(student_layers)] if student_layers>1 else [parent_layers-1]
        c.layers=student_layers;state={k:v for k,v in source.items() if not k.startswith('blocks.')}
        for new,old in enumerate(retained):
            prefix=f'blocks.{old}.'
            state.update({f'blocks.{new}.'+k[len(prefix):]:v for k,v in source.items() if k.startswith(prefix)})
    else:state=source
    backbone=LegalLM(c);backbone.load_state_dict(state,strict=True)
    parent_parameters=sum(p.numel() for p in source.values())
    if not allow_tiny and not 50_000_000<=parent_parameters<=300_000_000:raise ValueError('Phase-1 parent violates parameter bounds.')
    return backbone,{'phase1_sha256':file_hash(path),'parent_parameters':parent_parameters,
                     'retained_parent_blocks':retained,'phase1_model_config':saved['model_config']}

class NERModel(nn.Module):
    def __init__(self,backbone,cfg,labels,char_classes):
        super().__init__();self.backbone=backbone;self.cfg=cfg;self.labels=labels;self.char_classes=char_classes
        method=cfg['method'];dim=backbone.config.dim
        backbone.requires_grad_(method in ('full','kd'))
        if method=='partial':
            for block in list(backbone.blocks)[-cfg.get('partial_layers',3):]:block.requires_grad_(True)
            backbone.final_norm.requires_grad_(True)
        elif method in ('lora','lorma'):
            targets=cfg.get('projection_targets',['q','v','out','gate','up','down'])
            replacements=[(name,module) for name,module in backbone.named_modules() if isinstance(module,nn.Linear) and name.rsplit('.',1)[-1] in targets]
            for name,module in replacements:
                parent_name,attribute=name.rsplit('.',1);parent=backbone.get_submodule(parent_name)
                setattr(parent,attribute,LowRankLinear(module,cfg.get('rank',8),cfg.get('alpha',16),method=='lorma'))
        elif method=='adapter':
            for index,block in enumerate(backbone.blocks):backbone.blocks[index]=AdapterBlock(block,dim,cfg.get('adapter_dim',64))
            for name,p in backbone.named_parameters():
                if 'attn_norm' in name or 'ff_norm' in name or name.startswith('final_norm.'):p.requires_grad_(True)
        elif method not in ('full','kd','partial'):raise ValueError(method)
        self.attention_modules=[module for module in backbone.modules() if isinstance(module,Attention)]
        for module in self.attention_modules:module.forward=types.MethodType(ner_attention,module)
        self.two_directions=cfg.get('two_directions',True) and backbone.config.architecture!='encoder_decoder'
        width=dim*(2 if self.two_directions else 1)
        self.dropout=nn.Dropout(cfg.get('head_dropout',.1))
        self.feature_norm=nn.LayerNorm(width)
        self.left=nn.Linear(width,len(labels)*char_classes);self.right=nn.Linear(width,len(labels)*char_classes)
        if cfg['head']=='bio':self.classifier=nn.Linear(width,1+2*len(labels))
        else:
            inner=cfg.get('span_dim',32)
            self.span_projection=nn.Linear(width,len(labels)*2*inner)
            self.span_rotary=Rotary(inner,backbone.config.max_positions)
        # Heads are new task parameters; the backbone was loaded before they were created.
    def features(self,ids,valid):
        mask=None if bool(valid.all()) else valid
        for module in self.attention_modules:module._ner_valid=mask
        if self.backbone.config.architecture=='encoder_decoder':hidden=self.backbone(ids,source_ids=ids)
        else:
            hidden=self.backbone(ids)
            if self.two_directions:
                positions=torch.arange(ids.size(1),device=ids.device)[None,:].expand_as(ids)
                lengths=valid.sum(1,keepdim=True)
                reverse=torch.where(positions<lengths,lengths-1-positions,positions)
                reverse_ids=ids.gather(1,reverse)
                backward=self.backbone(reverse_ids).gather(1,reverse[:,:,None].expand(-1,-1,hidden.size(-1)))
                hidden=torch.cat((hidden,backward),-1)
        return self.dropout(self.feature_norm(hidden))
    def forward(self,ids,valid):
        hidden=self.features(ids,valid);b,t,_=hidden.shape;L=len(self.labels);C=self.char_classes
        result={'left':self.left(hidden).view(b,t,L,C),'right':self.right(hidden).view(b,t,L,C)}
        if self.cfg['head']=='bio':result['logits']=self.classifier(hidden)
        else:
            d=self.cfg.get('span_dim',32);pairs=self.span_projection(hidden).view(b,t,L,2,d)
            q=pairs[:,:,:,0].transpose(1,2);k=pairs[:,:,:,1].transpose(1,2)
            q=self.span_rotary(q);k=self.span_rotary(k)
            scores=torch.einsum('blid,bljd->blij',q,k)/math.sqrt(d)
            start=torch.arange(t,device=ids.device)[:,None];end=torch.arange(t,device=ids.device)[None,:]
            allowed=(end>=start)&(end-start<self.cfg.get('max_span_tokens',t))
            allowed=allowed[None,None]&valid[:,None,:,None]&valid[:,None,None,:]
            result['logits']=scores.masked_fill(~allowed,-1e4);result['span_valid']=allowed
        return result

def build_model(cfg,labels,char_classes,tokenizer_hash,restore=None):
    backbone,provenance=load_foundation(cfg['phase1_checkpoint'],tokenizer_hash,cfg['architecture'],
        cfg.get('student_layers') if cfg['method']=='kd' else None,cfg.get('allow_tiny',False))
    backbone.config.gradient_checkpointing=cfg.get('gradient_checkpointing',False)
    model=NERModel(backbone,cfg,labels,char_classes)
    if restore:model.load_state_dict(torch.load(restore,map_location='cpu',weights_only=False)['model'],strict=True)
    return model,provenance

def safe_ce(scores,targets):
    if bool((targets!=-100).any()):return F.cross_entropy(scores.float().reshape(-1,scores.size(-1)),targets.reshape(-1),ignore_index=-100)
    return scores.float().sum()*0

def ner_loss(result,batch,cfg,weights):
    if cfg['head']=='bio':
        hard=F.cross_entropy(result['logits'].float().reshape(-1,result['logits'].size(-1)),
            batch['tags'].reshape(-1),weight=weights,ignore_index=-100)
    else:
        scores=result['logits'].float();target=torch.zeros_like(scores,dtype=torch.bool)
        for row,item in enumerate(batch['items']):
            for start,end,label in item['entities']:target[row,label,start,end]=True
        # GlobalPointer Eq.13. Separate positive/negative logsumexp terms per label.
        valid=result['span_valid'].expand_as(scores)
        negative=scores.masked_fill(target|~valid,-1e4).flatten(2)
        positive=(-scores).masked_fill(~target|~valid,-1e4).flatten(2)
        zero=torch.zeros(*negative.shape[:2],1,device=scores.device)
        hard=(torch.logsumexp(torch.cat((negative,zero),-1),-1)+torch.logsumexp(torch.cat((positive,zero),-1),-1)).mean()
    boundary=safe_ce(result['left'],batch['left'])+safe_ce(result['right'],batch['right'])
    loss=hard+cfg.get('boundary_weight',.25)*boundary
    if batch.get('teacher') is not None:
        T=cfg.get('kd_temperature',2.);mask=batch['valid']
        student=result['logits'].float()[mask];teacher=batch['teacher'][mask]
        kl=F.kl_div(F.log_softmax(student/T,-1),F.softmax(teacher/T,-1),reduction='batchmean')*T*T
        alpha=cfg.get('kd_alpha',.5);loss=(1-alpha)*hard+alpha*kl+cfg.get('boundary_weight',.25)*boundary
    return loss

def parameter_report(model):
    return {'total_parameters':sum(p.numel() for p in model.parameters()),
            'trainable_parameters':sum(p.numel() for p in model.parameters() if p.requires_grad),
            'backbone_parameters':sum(p.numel() for p in model.backbone.parameters())}
'''
(CODE / 'ner_models.py').write_text(MODULE_SOURCE.lstrip('\n'), encoding='utf-8')
print('Wrote', 'ner_models.py')

In [ ]:
MODULE_SOURCE = r'''
"""Long-document inference, fixed BIO constraints, character boundaries, teacher caches."""
import argparse, contextlib, json, os, time, zipfile
from pathlib import Path
import numpy as np
import torch
import torch.distributed as dist
from ner_data import NERWindows, collate_windows, atomic_json, file_hash, digest, validate_spans, strict_metrics
from ner_models import build_model

def device_setup():
    rank=int(os.environ.get('RANK',0));local=int(os.environ.get('LOCAL_RANK',0));world=int(os.environ.get('WORLD_SIZE',1))
    device=torch.device('cuda',local) if torch.cuda.is_available() else torch.device('cpu')
    if device.type=='cuda':torch.cuda.set_device(local)
    if world>1 and not dist.is_initialized():dist.init_process_group('nccl' if device.type=='cuda' else 'gloo')
    return rank,world,device

def autocast(device):
    return torch.autocast('cuda',dtype=torch.float16) if device.type=='cuda' else contextlib.nullcontext()

def bio_viterbi(logits):
    """Fixed legality constraints, not a learned CRF. Input: tokens x (O,B1,I1,...)."""
    n,c=logits.shape
    if not n:return []
    transition=np.zeros((c,c),dtype=np.float32)
    for curr in range(2,c,2):
        transition[:,curr]=-1e9;transition[curr-1,curr]=0;transition[curr,curr]=0
    score=logits[0].copy();score[2::2]=-1e9;back=[]
    for row in logits[1:]:
        options=score[:,None]+transition;previous=options.argmax(0)
        score=options[previous,np.arange(c)]+row;back.append(previous)
    tag=int(score.argmax());tags=[tag]
    for previous in reversed(back):tag=int(previous[tag]);tags.append(tag)
    return tags[::-1]

def character_span(offsets,i,j,label,left,right,labels):
    a,b=offsets[i];c,d=offsets[j]
    # The boundary head predicts trims within the original BPE token's character span.
    l=int(np.argmax(left[i,label,:b-a]));r=int(np.argmax(right[j,label,:d-c]))
    start,end=a+l,d-r
    return [start,end,labels[label]] if start<end else None

def decode_bio(logits,offsets,left,right,labels):
    tags=bio_viterbi(logits);spans=[];i=0
    while i<len(tags):
        tag=tags[i]
        if tag==0:i+=1;continue
        label=(tag-1)//2;j=i
        while j+1<len(tags) and tags[j+1]==2+2*label:j+=1
        span=character_span(offsets,i,j,label,left,right,labels)
        if span is not None:spans.append(span)
        i=j+1
    return sorted({tuple(s) for s in spans})

@torch.inference_mode()
def predict_documents(model,cache,doc_ids,batch_size,device,threshold=0.,progress=None):
    model.eval();predictions={};labels=cache['labels'];L=len(labels);C=cache['char_classes']
    for did in doc_ids:
        if progress is not None and did in progress:predictions[did]=progress[did];continue
        doc=cache['encoded'][did];n=len(doc['ids'])
        if not n:predictions[did]=[];continue
        # One document at a time bounds RAM; all windows cover the entire input.
        windows=[w for w in cache['test_windows']+cache['all_windows'] if w['did']==did]
        local=dict(cache);local['infer_windows']=windows;ds=NERWindows(local,'infer',model.cfg['head'])
        if model.cfg['head']=='bio':
            emissions=np.zeros((n,1+2*L),np.float32);left=np.zeros((n,L,C),np.float32);right=np.zeros_like(left);counts=np.zeros(n,np.float32)
        candidates={}
        for start in range(0,len(ds),batch_size):
            batch=collate_windows([ds[i] for i in range(start,min(start+batch_size,len(ds)))])
            with autocast(device):out=model(batch['ids'].to(device),batch['valid'].to(device))
            values={k:v.float().cpu().numpy() for k,v in out.items() if k!='span_valid'}
            for row,item in enumerate(batch['items']):
                w=item['window'];a,b=w['start'],w['end'];length=b-a
                weight=np.minimum(np.arange(1,length+1),np.arange(length,0,-1)).astype(np.float32)
                weight=np.minimum(weight,cache['overlap']+1)
                if model.cfg['head']=='bio':
                    emissions[a:b]+=values['logits'][row,:length]*weight[:,None]
                    left[a:b]+=values['left'][row,:length]*weight[:,None,None];right[a:b]+=values['right'][row,:length]*weight[:,None,None]
                    counts[a:b]+=weight
                else:
                    scores=values['logits'][row,:,:length,:length];hits=np.argwhere(scores>threshold)
                    # A computational guard, recorded in config, rather than inventing entities.
                    limit=model.cfg.get('max_spans_per_window',1024)
                    if len(hits)>limit:
                        hit_scores=scores[tuple(hits.T)];hits=hits[np.argpartition(hit_scores,-limit)[-limit:]]
                    for label,i,j in hits:
                        span=character_span(item['offsets'],int(i),int(j),int(label),values['left'][row],values['right'][row],labels)
                        if span is not None:
                            key=tuple(span);candidates[key]=max(candidates.get(key,-1e9),float(scores[label,i,j]))
        if model.cfg['head']=='bio':
            if np.any(counts==0):raise ValueError('Inference omitted tokens.')
            spans=decode_bio(emissions/counts[:,None],doc['offsets'],left/counts[:,None,None],right/counts[:,None,None],labels)
        else:spans=sorted(candidates)
        predictions[did]=[list(s) for s in spans]
        if progress is not None:progress[did]=predictions[did]
    return predictions

@torch.inference_mode()
def cache_teacher(model,cache,split,output,device,batch_size,checkpoint_hash):
    if split not in ('train','all') or model.cfg['head']!='bio':raise ValueError('KD uses only labelled training windows and a BIO teacher.')
    output=Path(output);output.mkdir(parents=True,exist_ok=True)
    identity={'cache':cache['fingerprint'],'teacher_sha256':checkpoint_hash,'split':split,'windows':digest(cache[split+'_windows'])}
    manifest=output/'manifest.json'
    if manifest.exists() and json.loads(manifest.read_text())['identity']!=identity:raise ValueError('Teacher cache identity mismatch.')
    atomic_json(manifest,{'identity':identity,'complete':False});ds=NERWindows(cache,split,'bio');model.eval()
    for start in range(0,len(ds),batch_size):
        indices=[i for i in range(start,min(start+batch_size,len(ds))) if not (output/f'{i:06d}.npz').exists()]
        if not indices:continue
        batch=collate_windows([ds[i] for i in indices])
        with autocast(device):logits=model(batch['ids'].to(device),batch['valid'].to(device))['logits'].half().cpu().numpy()
        for row,index in enumerate(indices):
            target=output/f'{index:06d}.npz';temp=target.with_suffix('.tmp.npz')
            np.savez_compressed(temp,logits=logits[row,:len(batch['items'][row]['ids'])]);temp.replace(target)
    for i in range(len(ds)):
        values=np.load(output/f'{i:06d}.npz')['logits']
        if values.shape!=(len(ds[i]['ids']),1+2*len(cache['labels'])) or not np.isfinite(values).all():raise ValueError('Incomplete/invalid teacher logits.')
    atomic_json(manifest,{'identity':identity,'complete':True,'windows':len(ds)})

def submission_zip(predictions,texts,labels,path):
    validate_spans(predictions,texts,labels);path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    data=json.dumps(predictions,ensure_ascii=False,sort_keys=True,separators=(',',':')).encode()
    temp=path.with_suffix('.tmp.zip')
    with zipfile.ZipFile(temp,'w',zipfile.ZIP_DEFLATED) as archive:
        info=zipfile.ZipInfo('prediction.json',date_time=(2026,10,1,0,0,0));info.compress_type=zipfile.ZIP_DEFLATED
        archive.writestr(info,data)
    temp.replace(path)
    with zipfile.ZipFile(path) as archive:
        assert archive.namelist()==['prediction.json'];validate_spans(json.loads(archive.read('prediction.json')),texts,labels)
    return digest(predictions)

def main():
    p=argparse.ArgumentParser();p.add_argument('--checkpoint',required=True);p.add_argument('--cache',required=True)
    p.add_argument('--output',required=True);p.add_argument('--split',choices=['test','val','train','all'],default='test')
    p.add_argument('--teacher-cache',action='store_true');p.add_argument('--batch-size',type=int,default=4)
    args=p.parse_args();rank,world,device=device_setup()
    cache=torch.load(args.cache,map_location='cpu',weights_only=False);saved=torch.load(args.checkpoint,map_location='cpu',weights_only=False)
    model,_=build_model(saved['config'],cache['labels'],cache['char_classes'],cache['tokenizer_sha256'],args.checkpoint);model.to(device)
    if saved['cache_fingerprint']!=cache['fingerprint']:raise ValueError('Prediction checkpoint/data mismatch.')
    if args.teacher_cache:
        if rank==0:cache_teacher(model,cache,args.split,args.output,device,args.batch_size,file_hash(args.checkpoint))
    else:
        ids=cache['split']['val_ids'] if args.split=='val' else sorted({w['did'] for w in cache[args.split+'_windows']})
        if args.split=='test':ids=[k for k in cache['texts'] if k not in cache['gold']]
        root=Path(args.output);root.mkdir(parents=True,exist_ok=True);partial=root/f'part_{rank}.json'
        identity=digest([file_hash(args.checkpoint),cache['fingerprint'],args.split,ids,world])
        old=json.loads(partial.read_text()) if partial.exists() else {'identity':identity,'predictions':{}}
        if old['identity']!=identity:raise ValueError('Prediction resume identity changed; use a new output directory.')
        progress=old['predictions']
        for did in ids[rank::world]:
            progress.update(predict_documents(model,cache,[did],args.batch_size,device,progress=progress))
            atomic_json(partial,{'identity':identity,'predictions':progress})
        if world>1:dist.barrier()
        if rank==0:
            predictions={}
            for i in range(world):predictions.update(json.loads((root/f'part_{i}.json').read_text())['predictions'])
            texts={k:cache['texts'][k] for k in ids};validate_spans(predictions,texts,cache['labels'])
            atomic_json(root/'prediction.json',predictions)
            if args.split=='test':submission_zip(predictions,texts,cache['labels'],root/'submission.zip')
            if args.split=='val':atomic_json(root/'metrics.json',strict_metrics(predictions,{k:cache['gold'][k] for k in ids},cache['labels']))
    if world>1:dist.barrier();dist.destroy_process_group()

if __name__=='__main__':main()
'''
(CODE / 'ner_predict.py').write_text(MODULE_SOURCE.lstrip('\n'), encoding='utf-8')
print('Wrote', 'ner_predict.py')

In [ ]:
MODULE_SOURCE = r'''
"""Two-GPU DDP/FP16 training with strict-span selection and paired all-label refits."""
import argparse, contextlib, functools, json, math, os, random, time
from pathlib import Path
import numpy as np
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel
from torch.utils.data import DataLoader, DistributedSampler
from ner_data import NERWindows,collate_windows,class_weights,digest,atomic_json,strict_metrics
from ner_models import build_model,ner_loss,parameter_report
from ner_predict import device_setup,autocast,predict_documents

def atomic_torch(path,obj):
    path=Path(path);temp=path.with_name(path.name+'.tmp');torch.save(obj,temp);temp.replace(path)

def rng_state(device):
    return {'python':random.getstate(),'numpy':np.random.get_state(),'torch':torch.get_rng_state(),
            'cuda':torch.cuda.get_rng_state(device) if device.type=='cuda' else None}

def restore_rng(state,device):
    random.setstate(state['python']);np.random.set_state(state['numpy']);torch.set_rng_state(state['torch'])
    if device.type=='cuda':torch.cuda.set_rng_state(state['cuda'],device)

def main():
    p=argparse.ArgumentParser();p.add_argument('--config',required=True);p.add_argument('--cache',required=True)
    p.add_argument('--output',required=True);p.add_argument('--split',choices=['train','all'],default='train')
    p.add_argument('--epochs',type=int);p.add_argument('--teacher-dir');args=p.parse_args()
    cfg=json.loads(Path(args.config).read_text());rank,world,device=device_setup();root=Path(args.output);root.mkdir(parents=True,exist_ok=True)
    seed=cfg.get('seed',230437);random.seed(seed+rank);np.random.seed(seed+rank);torch.manual_seed(seed+rank)
    cache=torch.load(args.cache,map_location='cpu',weights_only=False)
    if args.teacher_dir:
        manifest=json.loads((Path(args.teacher_dir)/'manifest.json').read_text())
        if not manifest['complete'] or manifest['identity']['split']!=args.split or manifest['identity']['cache']!=cache['fingerprint']:
            raise ValueError('KD cache is incomplete or uses the wrong split.')
        if manifest['identity']['windows']!=digest(cache[args.split+'_windows']):raise ValueError('Teacher window geometry changed.')
    if cfg['method']=='kd' and not args.teacher_dir:raise ValueError('KD experiment requires its own larger teacher cache.')
    dataset=NERWindows(cache,args.split,cfg['head'],args.teacher_dir)
    if not len(dataset):raise ValueError('Empty training split.')
    sampler=DistributedSampler(dataset,num_replicas=world,rank=rank,shuffle=True,seed=seed,drop_last=False)
    generator=torch.Generator()
    loader=DataLoader(dataset,batch_size=cfg['micro_batch'],sampler=sampler,num_workers=cfg.get('workers',0),
        pin_memory=device.type=='cuda',collate_fn=collate_windows,generator=generator)
    model,provenance=build_model(cfg,cache['labels'],cache['char_classes'],cache['tokenizer_sha256']);model.to(device)
    report=parameter_report(model)
    if cfg['method']=='kd':
        teacher_report=cfg['teacher_parameters']
        if teacher_report<=report['total_parameters']:raise ValueError('Teacher must be larger than the student.')
    backbone=[];head=[]
    for name,param in model.named_parameters():
        if param.requires_grad:(backbone if name.startswith('backbone.') else head).append(param)
    groups=[{'params':backbone,'lr':cfg['backbone_lr'],'base_lr':cfg['backbone_lr']},
            {'params':head,'lr':cfg['head_lr'],'base_lr':cfg['head_lr']}]
    optimizer=torch.optim.AdamW(groups,weight_decay=cfg.get('weight_decay',.01),fused=device.type=='cuda')
    scaler=torch.amp.GradScaler('cuda',enabled=device.type=='cuda');weights=class_weights(cache,args.split).to(device)
    epochs=args.epochs or cfg.get('epochs',12);accum=cfg.get('accumulation',4)
    updates_per_epoch=math.ceil(len(loader)/accum);planned=updates_per_epoch*epochs
    identity=digest({'cfg':cfg,'phase1_sha256':provenance['phase1_sha256'],'cache':cache['fingerprint'],'split':args.split,'epochs':epochs,'world':world,
        'teacher_cache':manifest['identity'] if args.teacher_dir else None})
    epoch=cursor=updates=skipped=bad_epochs=0;best_f1=-1.;best_epoch=0;history=[];elapsed_prior=0.
    latest=root/'latest.pt';best=root/'best.pt';finished=root/'summary.json'
    if finished.exists():
        saved_summary=json.loads(finished.read_text())
        if saved_summary['identity']!=identity:raise ValueError('Completed run configuration changed; use a new run name.')
        if saved_summary.get('complete') and best.exists():
            if rank==0:print('Already complete:',root,flush=True)
            if world>1:dist.destroy_process_group()
            return
    if latest.exists():
        state=torch.load(latest,map_location='cpu',weights_only=False)
        if state['identity']!=identity:raise ValueError('Resume requires identical data, config, teacher, epochs and GPU count.')
        model.load_state_dict(state['model']);optimizer.load_state_dict(state['optimizer']);scaler.load_state_dict(state['scaler'])
        epoch=state['epoch'];cursor=state['cursor'];updates=state['updates'];skipped=state['amp_skipped_updates']
        best_f1=state['best_f1'];best_epoch=state['best_epoch'];bad_epochs=state['bad_epochs'];history=state['history'];elapsed_prior=state['elapsed_seconds']
        restore_rng(state['rng'][rank],device)
    wrapped=DistributedDataParallel(model,device_ids=[device.index] if device.type=='cuda' else None,broadcast_buffers=False) if world>1 else model
    started=time.monotonic();stopped=False
    def checkpoint(next_epoch,next_cursor,metrics=None,is_best=False,complete=False):
        random_states=[None]*world
        if world>1:dist.all_gather_object(random_states,rng_state(device))
        else:random_states[0]=rng_state(device)
        if rank==0:
            light={'model':model.state_dict(),'config':cfg,'labels':cache['labels'],'char_classes':cache['char_classes'],
                'cache_fingerprint':cache['fingerprint'],'provenance':provenance,'parameters':report,'metrics':metrics,
                'stage':'selection' if args.split=='train' else 'all_label_refit','selected_epoch':best_epoch,
                'training_origin':'own_phase1_adapted','epoch':next_epoch,'updates':updates}
            if is_best:atomic_torch(best,light)
            state={**light,'identity':identity,'optimizer':optimizer.state_dict(),'scaler':scaler.state_dict(),
                'cursor':next_cursor,'rng':random_states,'amp_skipped_updates':skipped,'best_f1':best_f1,'best_epoch':best_epoch,
                'bad_epochs':bad_epochs,'history':history,'elapsed_seconds':elapsed_prior+time.monotonic()-started}
            atomic_torch(latest,state)
            summary={k:state[k] for k in ('identity','epoch','updates','amp_skipped_updates','best_f1','best_epoch','history','elapsed_seconds')}
            summary.update({'parameters':report,'provenance':provenance,'complete':complete,'stop_reason':('time_cap' if stopped else 'epochs_or_patience') if complete else 'running','stage':light['stage']})
            atomic_json(finished,summary)
        if world>1:dist.barrier()
    if rank==0:print(json.dumps({'experiment':cfg['name'],'stage':args.split,'windows':len(dataset),'parameters':report,'effective_batch':world*cfg['micro_batch']*accum}),flush=True)
    while epoch<epochs:
        sampler.set_epoch(epoch);generator.manual_seed(seed+epoch);iterator=iter(loader)
        for _ in range(cursor):next(iterator)
        model.train();loss_total=0.;batches=0;epoch_start=time.monotonic()
        while cursor<len(loader):
            micro=[]
            for _ in range(min(accum,len(loader)-cursor)):micro.append(next(iterator))
            optimizer.zero_grad(set_to_none=True)
            warmup=max(1,round(planned*.1));step=updates+1
            factor=step/warmup if step<=warmup else .1+.9*.5*(1+math.cos(math.pi*min(1,(step-warmup)/max(planned-warmup,1))))
            for group in optimizer.param_groups:group['lr']=group['base_lr']*factor
            denominator=sum(len(batch['items']) for batch in micro)
            for mi,batch in enumerate(micro):
                for key in ('ids','valid','tags','left','right','teacher'):
                    if batch[key] is not None:batch[key]=batch[key].to(device,non_blocking=True)
                sync=wrapped.no_sync() if world>1 and mi+1<len(micro) else contextlib.nullcontext()
                with sync:
                    with autocast(device):
                        out=wrapped(batch['ids'],batch['valid']);loss=ner_loss(out,batch,cfg,weights)
                        if not bool(torch.isfinite(loss)):raise FloatingPointError('Non-finite forward loss; inspect the saved log and FP16/model stability before continuing.')
                        scaled_loss=loss*len(batch['items'])/denominator
                    scaler.scale(scaled_loss).backward()
                loss_total+=float(loss.detach());batches+=1
            scaler.unscale_(optimizer);norm=torch.nn.utils.clip_grad_norm_(model.parameters(),cfg.get('grad_clip',1.))
            scale_before=scaler.get_scale();scaler.step(optimizer);scaler.update()
            if scaler.get_scale()<scale_before:skipped+=1
            else:updates+=1
            cursor+=len(micro)
            flag=torch.tensor(int(time.monotonic()-started>=cfg.get('max_stage_hours',1.)*3600),device=device)
            if world>1:dist.all_reduce(flag,op=dist.ReduceOp.MAX)
            if bool(flag):stopped=True
            if updates and updates%cfg.get('checkpoint_every',50)==0 or stopped:checkpoint(epoch,cursor)
            if rank==0 and updates%cfg.get('log_every',20)==0:
                print(json.dumps({'epoch':epoch+1,'batch':cursor,'updates':updates,'loss':loss_total/max(batches,1),
                    'gradient_norm':float(norm),'elapsed_hours':(elapsed_prior+time.monotonic()-started)/3600,'amp_skipped_updates':skipped}),flush=True)
            if stopped:break
        metrics=None
        if world>1:dist.barrier()
        if args.split=='train' and rank==0:
            ids=cache['split']['val_ids'];pred=predict_documents(model,cache,ids,cfg.get('inference_batch',cfg['micro_batch']),device)
            metrics=strict_metrics(pred,{k:cache['gold'][k] for k in ids},cache['labels']);atomic_json(root/'validation_prediction.json',pred)
            print(json.dumps({'epoch':epoch+1,'strict_macro_f1':metrics['macro_f1']}),flush=True)
        if world>1:
            box=[metrics];dist.broadcast_object_list(box,src=0);metrics=box[0]
        improved=args.split=='all' or metrics['macro_f1']>best_f1+1e-7
        if improved:
            best_f1=metrics['macro_f1'] if metrics else -1.;best_epoch=epoch+1;bad_epochs=0
        else:bad_epochs+=1
        history.append({'epoch':epoch+1,'updates':updates,'metrics':metrics,'partial_epoch':cursor<len(loader),
                        'epoch_seconds':time.monotonic()-epoch_start})
        epoch+=1;cursor=0
        complete=stopped or epoch>=epochs or (args.split=='train' and bad_epochs>=cfg.get('patience',3))
        checkpoint(epoch,cursor,metrics,improved,complete)
        if complete:break
    if world>1:dist.destroy_process_group()

if __name__=='__main__':main()
'''
(CODE / 'train_ner.py').write_text(MODULE_SOURCE.lstrip('\n'), encoding='utf-8')
print('Wrote', 'train_ner.py')

In [ ]:
MODULE_SOURCE = r'''
"""Restore own foundations, retrieve official data, run seven auditable experiments."""
import csv, gc, json, os, shutil, subprocess, sys, time, zipfile
from pathlib import Path
import torch
from ner_data import atomic_json,file_hash,load_official,create_cache,read_json,digest
from ner_predict import submission_zip

ARCHITECTURES=('decoder','encoder_decoder','local_gqa','hybrid')
REPO='Exploration-Lab/CS-781-Capstone'
REVISION='89edba6a44d6bb9593fd13b6895b5cd3b06eb82e'
DATA_FILES=('Train/fold1-text.json','Train/fold1-gold.json','Train/fold2-text.json','Train/fold2-gold.json','Train/labels.txt','Test/fold3-text.json')

DEFAULT_PHASE1_ROOT='/kaggle/input/datasets/hardik4tiwari/phase-1-outputs/capstone_phase1'

def restore_foundations(input_root,output,overrides=None,tokenizer_override=None,phase1_root=None):
    """Load five explicit files from the direct capstone_phase1 directory only.

    input_root remains in the signature for compatibility; it is never searched.
    No ZIP, alternate output tree, pilot or benchmark is inspected.
    """
    selected_root=Path(phase1_root or DEFAULT_PHASE1_ROOT)
    if 'all_kaggle_outputs' in selected_root.parts:
        raise ValueError(f'Use the direct capstone_phase1 input directory: {DEFAULT_PHASE1_ROOT}')
    if not selected_root.is_dir():raise FileNotFoundError(f'Phase-1 input folder does not exist: {selected_root}')
    overrides=overrides or {}
    if set(overrides)-set(ARCHITECTURES):raise ValueError('Unknown architecture in FOUNDATION_OVERRIDES.')
    chosen={arch:str(Path(overrides.get(arch,selected_root/'runs'/f'main_{arch}'/'best_model.pt'))) for arch in ARCHITECTURES}
    source_tokenizer=Path(tokenizer_override or selected_root/'tokenizer.json')
    required=[Path(path) for path in chosen.values()]+[source_tokenizer]
    for path in required:
        if not path.resolve().is_relative_to(selected_root.resolve()):
            raise ValueError(f'Phase-1 input must come from {selected_root}: {path}')
    missing=[str(path) for path in required if not path.is_file()]
    if missing:raise FileNotFoundError('Missing Phase-1 checkpoint/tokenizer files: '+', '.join(missing))
    tokenizer_hash=file_hash(source_tokenizer)
    for arch,path in chosen.items():
        saved=torch.load(path,map_location='cpu',weights_only=False)
        if saved.get('model_config',{}).get('architecture')!=arch:
            raise ValueError(f'{path}: checkpoint architecture is not {arch}.')
        if saved.get('training_origin')!='random_initialization_official_phase1_only':
            raise ValueError(f'{path}: expected your own Phase-1 checkpoint.')
        if saved.get('tokenizer_sha256')!=tokenizer_hash:
            raise ValueError(f'{path}: checkpoint does not match the original tokenizer.')
        del saved;gc.collect()
    output=Path(output);output.mkdir(parents=True,exist_ok=True)
    tokenizer=output/'tokenizer.json'
    if source_tokenizer.resolve()!=tokenizer.resolve():shutil.copy2(source_tokenizer,tokenizer)
    source_checkpoints=dict(chosen)
    for arch,path in list(chosen.items()):
        target=output/'foundations'/arch/'best_model.pt';target.parent.mkdir(parents=True,exist_ok=True)
        if Path(path).resolve()!=target.resolve():
            if not target.exists() or file_hash(path)!=file_hash(target):shutil.copy2(path,target)
        chosen[arch]=str(target)
    atomic_json(output/'foundation_manifest.json',{'source_layout_root':str(selected_root),'input_mode':'direct_directory_only',
        'source_checkpoints':source_checkpoints,'source_tokenizer':str(source_tokenizer),
        'checkpoints':chosen,'tokenizer':str(tokenizer),'sha256':{a:file_hash(p) for a,p in chosen.items()},
        'tokenizer_sha256':tokenizer_hash})
    return chosen,str(tokenizer)

def obtain_data(project,input_root,local_root=None,revision=REVISION):
    destination=Path(project)/'data'
    if local_root is None:
        roots=[p.parent.parent for p in Path(input_root).rglob('fold1-text.json') if p.parent.name=='Train']
        roots=[p for p in roots if all((p/f).is_file() for f in DATA_FILES)]
        if len(roots)>1:raise ValueError('Multiple Phase-2 datasets found. Set DATA_ROOT explicitly.')
        local_root=roots[0] if roots else None
    if local_root:
        source=Path(local_root)
        if (source/'Phase_2_LNER').is_dir():source/= 'Phase_2_LNER'
        for name in DATA_FILES:
            target=destination/name;target.parent.mkdir(parents=True,exist_ok=True)
            if target.resolve()!=(source/name).resolve():shutil.copy2(source/name,target)
        origin={'source':'attached_official_data','source_root':str(source)}
    else:
        from huggingface_hub import hf_hub_download
        token=os.environ.get('HF_TOKEN')
        if not token:
            try:
                from kaggle_secrets import UserSecretsClient
                token=UserSecretsClient().get_secret('HF_TOKEN')
            except Exception:token=None
        for name in DATA_FILES:
            target=destination/name;target.parent.mkdir(parents=True,exist_ok=True)
            if target.exists():continue
            try:downloaded=hf_hub_download(REPO,'Phase_2_LNER/'+name,repo_type='dataset',revision=revision,token=token)
            except Exception as exc:raise RuntimeError('Official data could not be fetched. Accept the dataset terms and add an HF_TOKEN Kaggle secret, or attach its six Phase-2 files. Do not put a token in notebook code.') from exc
            shutil.copy2(downloaded,target)
        origin={'source':'official_huggingface','repo':REPO,'revision':revision}
    origin['sha256']={name:file_hash(destination/name) for name in DATA_FILES};atomic_json(destination/'provenance.json',origin)
    return str(destination)

def experiments(foundations,settings):
    base={'epochs':settings['epochs'],'micro_batch':settings['micro_batch'],'accumulation':settings['accumulation'],
        'head_lr':5e-4,'backbone_lr':5e-5,'max_stage_hours':settings['max_stage_hours'],'patience':3,'workers':0,
        'head_dropout':.1,'boundary_weight':.25,'two_directions':True,'seed':230437,'rank':8,'alpha':16,
        'gradient_checkpointing':settings.get('gradient_checkpointing',False),'checkpoint_every':50,'log_every':20,
        'inference_batch':settings['micro_batch'],'allow_tiny':settings.get('allow_tiny',False)}
    specs=[('01_decoder_full','decoder','full','bio'),('02_encoder_decoder_full','encoder_decoder','full','bio'),
        ('03_local_gqa_lora','local_gqa','lora','bio'),('04_hybrid_partial','hybrid','partial','bio'),
        ('05_decoder_lorma_span','decoder','lorma','span'),('06_decoder_adapter','decoder','adapter','bio'),
        ('07_decoder_student_kd','decoder','kd','bio')]
    rows=[]
    for name,arch,method,head in specs:
        row={**base,'name':name,'architecture':arch,'method':method,'head':head,'phase1_checkpoint':foundations[arch]}
        if method in ('lora','lorma','adapter'):row['backbone_lr']=2e-4
        if method=='partial':row['partial_layers']=3
        if method=='adapter':row['adapter_dim']=64
        if method=='lorma':row.update({'span_dim':32,'max_spans_per_window':1024})
        if method=='kd':row.update({'student_layers':settings.get('student_layers',4),'kd_temperature':2.,'kd_alpha':.5})
        rows.append(row)
    return rows

def launch(script,args,project,gpus=2,log=None):
    # Explicit loopback avoids hostname-resolution failures in isolated containers.
    import socket
    with socket.socket() as sock:sock.bind(('127.0.0.1',0));port=sock.getsockname()[1]
    code=Path(project)/'code';cmd=[sys.executable,'-m','torch.distributed.run','--nnodes=1','--master_addr=127.0.0.1',f'--master_port={port}',f'--nproc_per_node={gpus}',str(code/script),*map(str,args)]
    env=dict(os.environ);env['PYTHONPATH']=str(code)+os.pathsep+env.get('PYTHONPATH','');env['TOKENIZERS_PARALLELISM']='false'
    handle=Path(log).open('a',encoding='utf-8') if log else None
    try:
        proc=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1,env=env)
        for line in proc.stdout:
            print(line,end='',flush=True)
            if handle:handle.write(line);handle.flush()
        if proc.wait()!=0:raise RuntimeError(f'{script} failed; inspect {log or "cell output"}. Recovery checkpoints are retained.')
    finally:
        if handle:handle.close()

def run_experiments(project,rows,cache_path,settings):
    project=Path(project);cache=torch.load(cache_path,map_location='cpu',weights_only=False);catalog=[];gpus=settings.get('gpus',2)
    for cfg in rows:
        name=cfg['name'];root=project/'runs'/name;root.mkdir(parents=True,exist_ok=True)
        if cfg['method']=='kd':
            eligible=[c for c in catalog if c['head']=='bio']
            teacher=max(eligible,key=lambda c:c['validation_macro_f1'])
            cfg.update({'teacher_experiment':teacher['name'],'teacher_parameters':teacher['parameters']['total_parameters']})
            for stage,split in [('selection','train'),('refit','all')]:
                if stage=='refit' and not settings['refit_all']:continue
                checkpoint=project/'runs'/teacher['name']/stage/'best.pt'
                target=root/('teacher_'+split)
                launch('ner_predict.py',['--checkpoint',checkpoint,'--cache',cache_path,'--split',split,'--teacher-cache','--output',target,
                    '--batch-size',cfg['micro_batch']],project,gpus=1,log=root/f'teacher_{split}.log')
        config=root/'config.json';atomic_json(config,cfg)
        train_args=['--config',config,'--cache',cache_path,'--output',root/'selection','--split','train']
        if cfg['method']=='kd':train_args+=['--teacher-dir',root/'teacher_train']
        launch('train_ner.py',train_args,project,gpus,root/'selection.log')
        selection=root/'selection'/'best.pt';summary=read_json(root/'selection'/'summary.json')
        # Decode the selected checkpoint, not the last epoch, for comparable reported metrics.
        launch('ner_predict.py',['--checkpoint',selection,'--cache',cache_path,'--split','val','--output',root/'selected_validation',
            '--batch-size',cfg['micro_batch']],project,gpus,root/'validation.log')
        metrics=read_json(root/'selected_validation'/'metrics.json');test_checkpoint=selection
        if settings['refit_all']:
            refit_args=['--config',config,'--cache',cache_path,'--output',root/'refit','--split','all','--epochs',max(1,summary['best_epoch'])]
            if cfg['method']=='kd':refit_args+=['--teacher-dir',root/'teacher_all']
            launch('train_ner.py',refit_args,project,gpus,root/'refit.log');test_checkpoint=root/'refit'/'best.pt'
        launch('ner_predict.py',['--checkpoint',test_checkpoint,'--cache',cache_path,'--split','test','--output',root/'test_prediction',
            '--batch-size',cfg['micro_batch']],project,gpus,root/'prediction.log')
        prediction=read_json(root/'test_prediction'/'prediction.json');texts={k:cache['texts'][k] for k in prediction}
        zip_path=project/'submissions'/f'{name}.zip';pred_hash=submission_zip(prediction,texts,cache['labels'],zip_path)
        item={'name':name,'architecture':cfg['architecture'],'method':cfg['method'],'head':cfg['head'],'validation_macro_f1':metrics['macro_f1'],
            'metrics':metrics,'parameters':summary['parameters'],'selection_epoch':summary['best_epoch'],
            'selection_seconds':summary['elapsed_seconds'],'test_checkpoint':str(test_checkpoint),'refit_all':settings['refit_all'],
            'submission':str(zip_path),'prediction_sha256':pred_hash,'entities':sum(map(len,prediction.values()))}
        catalog.append(item);atomic_json(project/'catalog.json',catalog)
        print('READY TO DOWNLOAD:',zip_path,'held-out strict Macro-F1:',round(metrics['macro_f1'],5),flush=True)
    if len(catalog)!=7 or len({c['architecture'] for c in catalog})!=4:raise ValueError('Experiment coverage is incomplete.')
    hashes={c['prediction_sha256'] for c in catalog}
    if len(hashes)<7:
        atomic_json(project/'distinctness_warning.json',{'unique_predictions':len(hashes),'experiments':7})
        print('WARNING: Seven experiments ran, but some test predictions are identical. Renaming ZIPs does not create distinct predictions. Inspect per-label metrics and improve the affected experiment before uploading duplicates.')
    report=project/'comparison.csv'
    columns=['name','architecture','method','head','validation_macro_f1','selection_epoch','selection_seconds','entities','prediction_sha256']
    with report.open('w',newline='',encoding='utf-8') as f:
        writer=csv.DictWriter(f,fieldnames=columns);writer.writeheader();writer.writerows({k:c[k] for k in columns} for c in catalog)
    return catalog

def bundle_working(working,output):
    working=Path(working);output=Path(output);temp=output.with_suffix('.tmp.zip')
    files=[p for p in working.rglob('*') if p.is_file() and p.resolve() not in (output.resolve(),temp.resolve()) and '__pycache__' not in p.parts]
    total=sum(p.stat().st_size for p in files);free=shutil.disk_usage(working).free
    if free<total+256*1024**2:raise RuntimeError('Not enough free space for the complete working archive. Download the seven small submission ZIPs first; remove/re-export large recovery states deliberately, then rerun this cell.')
    with zipfile.ZipFile(temp,'w',zipfile.ZIP_DEFLATED,compresslevel=1,allowZip64=True) as z:
        for path in files:z.write(path,path.relative_to(working))
    temp.replace(output)
    with zipfile.ZipFile(output) as z:
        bad=z.testzip()
        if bad:raise ValueError(f'Archive CRC failure: {bad}')
    return str(output)

def benchmark_suite(rows,cache,project,settings):
    """Actual T4 memory/step measurement; no benchmark weights are used for submission."""
    import math
    from ner_data import NERWindows,collate_windows,class_weights
    from ner_models import build_model,ner_loss,parameter_report
    from ner_predict import autocast
    if not torch.cuda.is_available():raise RuntimeError('Run this preflight on Kaggle T4 GPUs.')
    device=torch.device('cuda',0);reports=[];effective=settings['micro_batch']*settings['accumulation']
    for cfg in rows:
        while True:
            torch.cuda.empty_cache();model=None;optimizer=None;batch=None;out=None;loss=None;params=None;scaler=None;weights=None
            try:
                model,_=build_model(cfg,cache['labels'],cache['char_classes'],cache['tokenizer_sha256']);model.to(device).train()
                params=[p for p in model.parameters() if p.requires_grad];optimizer=torch.optim.AdamW(params,lr=1e-4,fused=True)
                scaler=torch.amp.GradScaler('cuda');weights=class_weights(cache,'train').to(device)
                ds=NERWindows(cache,'train',cfg['head']);longest=max(range(len(ds)),key=lambda i:len(ds[i]['ids']))
                batch=collate_windows([ds[longest] for _ in range(cfg['micro_batch'])])
                for key in ('ids','valid','tags','left','right'):batch[key]=batch[key].to(device)
                torch.cuda.reset_peak_memory_stats();times=[]
                successful=attempts=0
                while successful<4 and attempts<16:
                    attempts+=1
                    torch.cuda.synchronize();start=time.monotonic();optimizer.zero_grad(set_to_none=True)
                    with autocast(device):out=model(batch['ids'],batch['valid']);loss=ner_loss(out,batch,cfg,weights)
                    if not bool(torch.isfinite(loss)):raise ValueError('Preflight loss is non-finite.')
                    scaler.scale(loss).backward();scaler.unscale_(optimizer);torch.nn.utils.clip_grad_norm_(model.parameters(),1.)
                    old_scale=scaler.get_scale();scaler.step(optimizer);scaler.update();torch.cuda.synchronize()
                    if scaler.get_scale()>=old_scale:
                        successful+=1
                        if successful>1:times.append(time.monotonic()-start)
                if successful<4:raise ValueError('Preflight could not obtain four finite FP16 updates after scale adjustment.')
                peak=torch.cuda.max_memory_reserved()/1024**3
                if peak>settings.get('memory_budget_gib',12.5):raise torch.cuda.OutOfMemoryError('Reserve headroom for DDP and full GPU execution.')
                mean=sum(times)/len(times);batches=math.ceil(math.ceil(len(ds)/settings['gpus'])/cfg['micro_batch'])
                estimate=mean*batches*cfg['epochs']*(2 if settings['refit_all'] else 1)
                reports.append({'name':cfg['name'],'micro_batch':cfg['micro_batch'],'accumulation':cfg['accumulation'],
                    'microbatch_seconds_single_gpu':mean,'peak_reserved_gib_single_gpu':peak,'parameters':parameter_report(model),
                    'rough_train_refit_hours_excluding_validation_ddp_io':estimate/3600})
                print(reports[-1],flush=True);break
            except torch.cuda.OutOfMemoryError:
                if cfg['micro_batch']==1:raise RuntimeError('Preflight still runs out of memory at micro-batch 1. Enable gradient checkpointing or reduce SEQ_LEN and rebuild the cache.')
                cfg['micro_batch']=max(1,cfg['micro_batch']//2);cfg['accumulation']=max(1,math.ceil(effective/cfg['micro_batch']))
                cfg['inference_batch']=cfg['micro_batch'];print('Reducing',cfg['name'],'micro-batch to',cfg['micro_batch'],flush=True)
            finally:
                del model,optimizer,batch,out,loss,params,scaler,weights;gc.collect();torch.cuda.empty_cache()
    atomic_json(Path(project)/'benchmark.json',reports)
    print('Rough measured training+refit forecast (excludes validation, KD caching, inference, DDP and IO):',round(sum(r['rough_train_refit_hours_excluding_validation_ddp_io'] for r in reports),2),'hours')
    return reports
'''
(CODE / 'phase2_workflow.py').write_text(MODULE_SOURCE.lstrip('\n'), encoding='utf-8')
print('Wrote', 'phase2_workflow.py')

In [ ]:
MODULE_SOURCE = r'''
"""Disposable two-update DDP check for every method on the actual Kaggle GPUs."""
import argparse,gc,json
from pathlib import Path
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel
from ner_predict import device_setup,autocast
from ner_models import build_model,ner_loss
from ner_data import NERWindows,collate_windows,class_weights,atomic_json

def main():
    p=argparse.ArgumentParser();p.add_argument('--plan',required=True);p.add_argument('--cache',required=True);p.add_argument('--output',required=True)
    args=p.parse_args();rank,world,device=device_setup()
    if world!=2 or device.type!='cuda':raise RuntimeError('This smoke test requires the actual two Kaggle GPUs.')
    cache=torch.load(args.cache,map_location='cpu',weights_only=False);rows=json.loads(Path(args.plan).read_text());reports=[]
    for cfg in rows:
        model,_=build_model(cfg,cache['labels'],cache['char_classes'],cache['tokenizer_sha256']);model.to(device).train()
        wrapped=DistributedDataParallel(model,device_ids=[device.index],broadcast_buffers=False)
        optimizer=torch.optim.AdamW([p for p in model.parameters() if p.requires_grad],lr=1e-4,fused=True)
        scaler=torch.amp.GradScaler('cuda');weights=class_weights(cache,'train').to(device);ds=NERWindows(cache,'train',cfg['head'])
        indices=sorted(range(len(ds)),key=lambda i:len(ds[i]['ids']),reverse=True)
        batch=collate_windows([ds[indices[(rank*cfg['micro_batch']+i)%len(indices)]] for i in range(cfg['micro_batch'])])
        for key in ('ids','valid','tags','left','right'):batch[key]=batch[key].to(device)
        successful=attempts=0
        while successful<2 and attempts<16:
            attempts+=1
            optimizer.zero_grad(set_to_none=True)
            with autocast(device):out=wrapped(batch['ids'],batch['valid']);loss=ner_loss(out,batch,cfg,weights)
            if not bool(torch.isfinite(loss)):raise ValueError('Non-finite loss during distributed preflight.')
            scaler.scale(loss).backward();scaler.unscale_(optimizer)
            norm=torch.nn.utils.clip_grad_norm_(model.parameters(),1.)
            old_scale=scaler.get_scale();scaler.step(optimizer);scaler.update()
            if scaler.get_scale()>=old_scale:successful+=1
        if successful<2:raise ValueError('FP16 preflight could not complete two finite updates after scale adjustment; inspect model and loss before long training.')
        dist.barrier();reports.append({'name':cfg['name'],'two_ddp_updates_passed':True,'attempted_updates':attempts})
        if rank==0:print('PASS actual GPU DDP:',cfg['name'],flush=True)
        del wrapped,model,optimizer,scaler,weights,batch,out,loss;gc.collect();torch.cuda.empty_cache()
    if rank==0:atomic_json(args.output,reports)
    dist.barrier();dist.destroy_process_group()

if __name__=='__main__':main()
'''
(CODE / 'ddp_smoke.py').write_text(MODULE_SOURCE.lstrip('\n'), encoding='utf-8')
print('Wrote', 'ddp_smoke.py')

In [ ]:
MODULE_SOURCE = r'''
#!/usr/bin/env python3
"""Local evaluator for Phase 2.

Usage:
  python evaluate_phase2.py --truth truth.json --prediction prediction.json --labels labels.txt
"""
import argparse, json, sys

class DuplicateJSONKeyError(ValueError): pass
def reject_constant(v): raise ValueError(f"Non-standard JSON numeric value '{v}' is not allowed.")
def pairs_no_duplicates(pairs):
    out={}; dup=[]
    for k,v in pairs:
        if k in out: dup.append(k)
        out[k]=v
    if dup: raise DuplicateJSONKeyError(f"Duplicate JSON object keys detected: {list(dict.fromkeys(dup))[:10]}.")
    return out
def load_json(path):
    with open(path,encoding='utf-8') as f:
        try: return json.load(f, object_pairs_hook=pairs_no_duplicates, parse_constant=reject_constant)
        except json.JSONDecodeError as e: raise ValueError(f"Invalid JSON in {path} at line {e.lineno}, column {e.colno}: {e.msg}") from e
def labels(path):
    with open(path,encoding='utf-8') as f: xs=[x.strip() for x in f if x.strip()]
    if not xs: raise ValueError('labels.txt is empty.')
    if len(xs)!=12: raise ValueError(f'Expected 12 labels, found {len(xs)}.')
    if len(set(xs))!=len(xs): raise ValueError('labels.txt contains duplicate labels.')
    return xs
def spans(x, allowed, where):
    if x is None: return set()
    if not isinstance(x,list): raise ValueError(f'{where}: expected a list of entities.')
    out=set()
    for j,s in enumerate(x):
        if not isinstance(s,list) or len(s) not in (3,4): raise ValueError(f'{where}, entity {j}: expected [start,end,label] or [start,end,label,entity_text].')
        a,b,l=s[:3]
        if isinstance(a,bool) or not isinstance(a,int) or isinstance(b,bool) or not isinstance(b,int): raise ValueError(f'{where}, entity {j}: start/end must be integers.')
        if a<0 or b<=a: raise ValueError(f'{where}, entity {j}: require 0 <= start < end.')
        if not isinstance(l,str) or l not in allowed: raise ValueError(f'{where}, entity {j}: invalid label {l!r}.')
        if len(s)==4 and not isinstance(s[3],str): raise ValueError(f'{where}, entity {j}: entity_text must be a string.')
        t=(a,b,l)
        if t in out: raise ValueError(f'{where}, entity {j}: duplicate span {t}.')
        out.add(t)
    return out

def main():
    ap=argparse.ArgumentParser(description='Evaluate Phase 2 entity predictions on your own held-out split.')
    ap.add_argument('--truth',required=True); ap.add_argument('--prediction',required=True); ap.add_argument('--labels',required=True)
    args=ap.parse_args(); truth=load_json(args.truth); pred=load_json(args.prediction); labs=labels(args.labels)
    if not isinstance(truth,dict) or not truth or not isinstance(pred,dict) or not pred: raise ValueError('Truth and prediction must be non-empty JSON objects.')
    if set(truth)!=set(pred): raise ValueError(f"ID mismatch: missing={len(set(truth)-set(pred))}, extra={len(set(pred)-set(truth))}")
    stats={l:[0,0,0] for l in labs}; allowed=set(labs)
    for did in truth:
        g=spans(truth[did],allowed,f'truth[{did}]'); p=spans(pred[did],allowed,f'prediction[{did}]')
        for l in labs:
            gs={x for x in g if x[2]==l}; ps={x for x in p if x[2]==l}
            stats[l][0]+=len(gs&ps); stats[l][1]+=len(ps-gs); stats[l][2]+=len(gs-ps)
    precisions=[]; recalls=[]; f1s=[]
    print(f'Examples: {len(truth)}')
    for l in labs:
        tp,fp,fn=stats[l]; pr=tp/(tp+fp) if tp+fp else 0; rc=tp/(tp+fn) if tp+fn else 0; f=2*pr*rc/(pr+rc) if pr+rc else 0
        precisions.append(pr); recalls.append(rc); f1s.append(f)
        print(f'{l}: P={pr:.6f} R={rc:.6f} F1={f:.6f}')
    print(f'Strict Macro-F1:        {sum(f1s)/len(f1s):.6f}')
    print(f'Strict Macro-Precision: {sum(precisions)/len(precisions):.6f}')
    print(f'Strict Macro-Recall:    {sum(recalls)/len(recalls):.6f}')
if __name__=='__main__':
    try: main()
    except Exception as e: print(f'EVALUATION FAILED: {e}',file=sys.stderr); sys.exit(1)
'''
(CODE / 'evaluate_phase2_official.py').write_text(MODULE_SOURCE.lstrip('\n'), encoding='utf-8')
print('Wrote', 'evaluate_phase2_official.py')

In [ ]:
from phase2_workflow import restore_foundations, obtain_data, experiments, benchmark_suite, run_experiments, bundle_working, launch
from ner_data import load_official, create_cache, atomic_json, validate_spans, file_hash, read_json, digest
from ner_predict import submission_zip, bio_viterbi, decode_bio
import importlib, phase2_workflow
importlib.invalidate_caches()
importlib.reload(phase2_workflow)  # Use the rewritten loader even in an existing Kaggle session.
from phase2_workflow import restore_foundations
foundations, tokenizer_path = restore_foundations(INPUT_ROOT, PROJECT / 'phase1', FOUNDATION_OVERRIDES, TOKENIZER_OVERRIDE,
                                                 phase1_root=PHASE1_ROOT)
print('Found all four own Phase-1 foundations:')
for architecture, path in foundations.items(): print(architecture, path)
print('Original tokenizer:', tokenizer_path)
input_manifest = read_json(PROJECT / 'phase1' / 'foundation_manifest.json')
print('Original Phase-1 output folder:', input_manifest['source_layout_root'])
print('Selected original checkpoints:', json.dumps(input_manifest['source_checkpoints'], indent=2))

## Official data and an honest document-level holdout

Both public training folds are pooled. Whole duplicate document groups stay in one partition. A deterministic label-presence heuristic attempts validation coverage while preserving training support, including singleton-label groups, when feasible. It may produce a smaller holdout to protect that support and reports any unavoidable missing labels. Small label support still limits conclusions.

Text is never stripped, normalized or reconstructed for prediction offsets. Native tokenizer offsets are Python character indexes. The audit checks every annotated span, counts BIO-overlap omissions and increases overlap if required for span containment. The GlobalPointer candidate preserves overlapping annotations. If an entity cannot fit, increase `SEQ_LEN` and choose a new cache path; do not drop it silently. Optional gold entity text is diagnostic only, matching the official scorer's treatment of that field.

In [ ]:
data_root = obtain_data(PROJECT, INPUT_ROOT, DATA_ROOT)
train_texts, gold, test_texts, labels, original_folds = load_official(data_root)
CACHE_PATH = PROJECT / f'ner_cache_v3_s{SEQ_LEN}_o{OVERLAP}_seed{SEED}.pt'
cache = create_cache(train_texts, gold, test_texts, labels, tokenizer_path, CACHE_PATH,
                     seq_len=SEQ_LEN, overlap=OVERLAP, seed=SEED, val_fraction=VALIDATION_FRACTION)
print('Train documents:', len(train_texts), 'Test documents:', len(test_texts))
print('Official label order:', labels)
print('Character / annotation audit:', json.dumps(cache['audit'], indent=2))
print('Document split:', json.dumps(cache['split'], indent=2))
print('Window counts:', {split: len(cache[split + '_windows']) for split in ['train', 'val', 'all', 'test']})
print('Effective overlap:', cache['overlap'], 'Boundary classes:', cache['char_classes'])
atomic_json(PROJECT / 'split.json', cache['split'])
atomic_json(PROJECT / 'annotation_audit.json', cache['audit'])
atomic_json(PROJECT / 'run_settings.json', {'settings': SETTINGS, 'seq_len': SEQ_LEN, 'requested_overlap': OVERLAP,
                                         'validation_fraction': VALIDATION_FRACTION, 'seed': SEED})

## Offset and evaluator check before costly training

This uses training annotations to verify the conversion pipeline, not to estimate model accuracy. Oracle token labels and boundary logits must reproduce every BIO-representable gold span exactly. It then compares the local metric with the official evaluator on the held-out gold itself; absent labels can make that oracle macro score less than one because all twelve labels enter the denominator.

In [ ]:
import numpy as np, re
from ner_data import strict_metrics
for did in train_texts:
    offsets = cache['encoded'][did]['offsets']; alignment = cache['aligned'][did]; n = len(offsets)
    if not n: continue
    scores = np.full((n, 1 + 2 * len(labels)), -100., dtype=np.float32)
    scores[np.arange(n), alignment['tags']] = 100.
    left = np.zeros((n, len(labels), cache['char_classes']), dtype=np.float32); right = left.copy()
    for start, end, label, a, b in alignment['bio_entities']:
        left[start, label, a] = 100.; right[end, label, b] = 100.
    recovered = {tuple(s) for s in decode_bio(scores, offsets, left, right, labels)}
    expected = {(offsets[i][0] + a, offsets[j][1] - b, labels[l])
                for i, j, l, a, b in alignment['bio_entities']}
    if recovered != expected: raise ValueError(f'Character-boundary round trip failed for {did}.')
    del scores, left, right
validation_ids = cache['split']['val_ids']
truth = {did: [entity[:3] for entity in gold[did]] for did in validation_ids}
check_dir = PROJECT / 'evaluator_check'; check_dir.mkdir(exist_ok=True)
atomic_json(check_dir / 'truth.json', truth)
atomic_json(check_dir / 'prediction.json', truth)
checked = subprocess.run([sys.executable, str(CODE / 'evaluate_phase2_official.py'),
    '--truth', str(check_dir / 'truth.json'), '--prediction', str(check_dir / 'prediction.json'),
    '--labels', str(Path(data_root) / 'Train/labels.txt')], check=True, capture_output=True, text=True)
official_score = float(re.search(r'Strict Macro-F1:\s+([0-9.]+)', checked.stdout).group(1))
assert abs(official_score - strict_metrics(truth, truth, labels)['macro_f1']) < 1e-6
print('PASS: every BIO-representable gold character boundary round-trips; official evaluator agrees.')
print(checked.stdout)

## Seven methods and measured T4 preflight

Preflight uses disposable model copies; their updated weights are discarded. It tests the longest training window, includes optimizer allocations, uses FP16 and leaves memory headroom for DDP. An OOM reduces the micro-batch and raises accumulation to preserve approximate effective batch size. Gradient checkpointing is supported but initially disabled because it adds recomputation.

The forecast uses single-GPU micro-batch timing and assumes ideal two-GPU division. **It excludes distributed overhead, validation, KD caching, test inference and archiving.** Treat it as an estimate, inspect observed epoch times, and lower budgets before the deadline if necessary. PyTorch SDPA chooses a supported kernel; FlashAttention-2 is not assumed on T4.

In [ ]:
rows = experiments(foundations, SETTINGS)
for row in rows:
    print(row['name'], row['architecture'], row['method'], row['head'])
if RUN_PREFLIGHT:
    measurements = benchmark_suite(rows, cache, PROJECT, SETTINGS)
atomic_json(PROJECT / 'experiment_plan.json', rows)
launch('ddp_smoke.py', ['--plan', PROJECT / 'experiment_plan.json', '--cache', CACHE_PATH,
                       '--output', PROJECT / 'actual_ddp_preflight.json'], PROJECT, gpus=2,
       log=PROJECT / 'actual_ddp_preflight.log')
print('The training cell below runs all seven in order and saves each completed candidate immediately.')

## Train, select, optionally refit, and generate seven prediction ZIPs

Epoch selection uses strict span Macro-F1 on held-out documents, with early stopping. Refit starts from the original Phase-1 foundation and uses all labelled documents for the selected epoch count; its test output carries the earlier selection score, not a refit validation score. Candidate 07 uses the best measured BIO teacher. The validation-stage teacher and its cached logits exclude validation documents. A separate all-label teacher cache is used during student refit.

Training uses two-GPU DDP, FP16 scaling, gradient accumulation, gradient clipping and atomic recovery checkpoints. Logs retain elapsed time, loss, parameter counts and skipped AMP updates. Exact GPU numerical reproducibility across different kernels is not promised. If a cell fails, inspect its log and retain the original settings for recovery. A time-capped stage is marked as such in its summary.

In [ ]:
catalog = run_experiments(PROJECT, rows, str(CACHE_PATH), SETTINGS)
print('Generated', len(catalog), 'submission candidates.')

In [ ]:
# Add this whole file as ONE new code cell after the seven-model training cell.
# Uses the original Phase-1 decoder only. No downloaded model weights.
# A score of 0.75-0.80 is a target, not an accuracy guarantee.
from pathlib import Path
import ast, gc, importlib, json, math, sys
import numpy as np
import torch
from IPython.display import FileLink, display

BOOST_CONTEXT = 768
BOOST_EPOCHS = 24
BOOST_PATIENCE = 5
BOOST_SEEDS = (230437, 230438)
BOOST_MAX_STAGE_HOURS = 0.50  # Separate cap for EACH selection/refit stage.
BOOST_REFIT = True
BOOST_ROOT_NAME = 'extra_span_v1'  # Change this if you change settings after training.

for variable in ('PROJECT', 'CODE', 'CACHE_PATH'):
    if variable not in globals():
        raise RuntimeError(f'Run the original notebook setup and data cells first: missing {variable}.')
if torch.cuda.device_count() != 2:
    raise RuntimeError('This cell is configured for Kaggle GPU T4 x2.')
PROJECT, CODE = Path(PROJECT), Path(CODE)
BOOST_ROOT = PROJECT / BOOST_ROOT_NAME
BOOST_ROOT.mkdir(parents=True, exist_ok=True)
foundation_root = Path('/kaggle/input/datasets/hardik4tiwari/phase-1-outputs/capstone_phase1')
foundation_checkpoint = foundation_root / 'runs/main_decoder/best_model.pt'
if not foundation_checkpoint.is_file():
    raise FileNotFoundError(f'Attach the direct capstone_phase1 folder: {foundation_checkpoint}')
if str(CODE) not in sys.path: sys.path.insert(0, str(CODE))

BOOST_MODULE = r'''
"""Own-foundation bidirectional span NER; imported only by the additional cell."""
import argparse, bisect, json, math, os, sys
from pathlib import Path
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
import torch.distributed as dist
from ner_models import NERModel, load_foundation, ner_loss as original_loss
from ner_data import NERWindows, collate_windows, windows_for, digest, atomic_json, file_hash
from ner_predict import autocast, character_span

def setup_device():
    rank=int(os.environ.get('RANK',0));local=int(os.environ.get('LOCAL_RANK',0));world=int(os.environ.get('WORLD_SIZE',1))
    device=torch.device('cuda',local) if torch.cuda.is_available() else torch.device('cpu')
    if device.type=='cuda':torch.cuda.set_device(local)
    if world>1 and not dist.is_initialized():
        if device.type=='cuda':dist.init_process_group('nccl',device_id=device)
        else:dist.init_process_group('gloo')
    return rank,world,device

class SpanWindows(NERWindows):
    def __getitem__(self,index):
        item=super().__getitem__(index);w=item['window']
        item['tags']=torch.from_numpy(self.cache['aligned'][w['did']]['tags'][w['start']:w['end']].copy())
        return item

class BidirectionalSpanNER(NERModel):
    def __init__(self,backbone,cfg,labels,char_classes):
        super().__init__(backbone,cfg,labels,char_classes)
        if backbone.config.architecture!='decoder' or self.two_directions or cfg['head']!='span':
            raise ValueError('This extension expects the original decoder, one bidirectional pass, and a span head.')
        for attention in self.attention_modules:
            attention.causal=False;attention.window=0;attention.dropout=cfg.get('backbone_dropout',.1)
        for module in self.backbone.modules():
            if isinstance(module,nn.Dropout):module.p=cfg.get('backbone_dropout',.1)
        self.aux_bio=nn.Linear(backbone.config.dim,1+2*len(labels))

    def forward(self,ids,valid):
        h=self.features(ids,valid);b,t,_=h.shape;L=len(self.labels);C=self.char_classes
        d=self.cfg.get('span_dim',64)
        pairs=self.span_projection(h).view(b,t,L,2,d)
        q=self.span_rotary(pairs[:,:,:,0].transpose(1,2))
        k=self.span_rotary(pairs[:,:,:,1].transpose(1,2))
        scores=torch.einsum('blid,bljd->blij',q,k)/math.sqrt(d)
        i=torch.arange(t,device=ids.device)[:,None];j=torch.arange(t,device=ids.device)[None,:]
        allowed=((j>=i)&(j-i<self.cfg.get('max_span_tokens',t)))[None,None]
        allowed=allowed&valid[:,None,:,None]&valid[:,None,None,:]
        return {'logits':scores.masked_fill(~allowed,-1e4),'span_valid':allowed,
                'left':self.left(h).view(b,t,L,C),'right':self.right(h).view(b,t,L,C),
                'aux_bio':self.aux_bio(h)}

def build_boost(cfg,labels,char_classes,tokenizer_hash,restore=None):
    backbone,provenance=load_foundation(cfg['phase1_checkpoint'],tokenizer_hash,'decoder',allow_tiny=cfg.get('allow_tiny',False))
    backbone.config.gradient_checkpointing=cfg.get('gradient_checkpointing',False)
    model=BidirectionalSpanNER(backbone,cfg,labels,char_classes)
    if restore:model.load_state_dict(torch.load(restore,map_location='cpu',weights_only=False)['model'],strict=True)
    return model,provenance

def boost_loss(result,batch,cfg,weights):
    aux=F.cross_entropy(result['aux_bio'].float().flatten(0,1),batch['tags'].flatten(),
                        weight=weights,ignore_index=-100,label_smoothing=.03)
    return original_loss(result,batch,cfg,weights)+cfg.get('aux_bio_weight',.3)*aux

def trim_whitespace(text,start,end):
    while start<end and text[start].isspace():start+=1
    while end>start and text[end-1].isspace():end-=1
    return start,end

@torch.inference_mode()
def raw_documents(model,cache,doc_ids,batch_size,device,floor=-2.):
    model.eval();result={};labels=cache['labels'];cap=model.cfg.get('candidate_cap',4096)
    for did in doc_ids:
        doc=cache['encoded'][did];text=cache['texts'][did]
        local=dict(cache);local['infer_windows']=windows_for(cache['encoded'],[did],cache['seq_len'],cache['overlap'])
        dataset=NERWindows(local,'infer','span');maps={'learned':{},'whitespace':{}}
        for first in range(0,len(dataset),batch_size):
            batch=collate_windows([dataset[j] for j in range(first,min(first+batch_size,len(dataset)))])
            with autocast(device):out=model(batch['ids'].to(device),batch['valid'].to(device))
            for row,item in enumerate(batch['items']):
                n=len(item['ids']);scores=out['logits'][row,:,:n,:n].float()
                hits=(scores>floor).nonzero()
                if hits.size(0)>cap:
                    values=scores[hits[:,0],hits[:,1],hits[:,2]]
                    hits=hits[values.topk(cap).indices]
                values=scores[hits[:,0],hits[:,1],hits[:,2]].cpu().tolist()
                hits=hits.cpu().tolist()
                left=out['left'][row,:n].float().cpu().numpy();right=out['right'][row,:n].float().cpu().numpy()
                offsets=item['offsets']
                for (label,i,j),score in zip(hits,values):
                    span=character_span(offsets,i,j,label,left,right,labels)
                    if span is not None:
                        key=tuple(span);maps['learned'][key]=max(maps['learned'].get(key,-1e9),score)
                    a,b=trim_whitespace(text,offsets[i][0],offsets[j][1])
                    if a<b:
                        key=(a,b,labels[label]);maps['whitespace'][key]=max(maps['whitespace'].get(key,-1e9),score)
        result[did]={mode:[[*key,float(value)] for key,value in sorted(candidates.items())] for mode,candidates in maps.items()}
    return result

def suppress(spans,allow_nested=False):
    unique={}
    for s in spans:
        key=tuple(s[:3]);unique[key]=max(unique.get(key,-1e9),float(s[3]))
    ordered=sorted(unique,key=lambda key:(-unique[key],key[1]-key[0],key))
    if allow_nested:return sorted(ordered)
    chosen=[];starts=[];ends=[]
    for key in ordered:
        a,b,_=key;pos=bisect.bisect_left(starts,a)
        if (pos and ends[pos-1]>a) or (pos<len(starts) and starts[pos]<b):continue
        starts.insert(pos,a);ends.insert(pos,b);chosen.append(key)
    return sorted(chosen)

def training_nested(cache,ids):
    flags={label:False for label in cache['labels']}
    for did in ids:
        for label in cache['labels']:
            spans=sorted({tuple(e[:2]) for e in cache['gold'][did] if e[2]==label})
            end=-1
            for a,b in spans:
                if a<end:flags[label]=True
                end=max(end,b)
    return flags

def decode_raw(raw,labels,calibration):
    mode=calibration['boundary_mode'];thresholds=calibration['thresholds'];nested=calibration['allow_nested']
    predictions={}
    for did,options in raw.items():
        rows=options[mode];entities=[]
        for label in labels:
            candidates=[s for s in rows if s[2]==label and s[3]>thresholds[label]]
            entities.extend(suppress(candidates,nested.get(label,False)))
        predictions[did]=[list(e) for e in sorted(set(entities))]
    return predictions

def default_calibration(cache):
    return {'boundary_mode':'learned','thresholds':{label:0. for label in cache['labels']},
            'allow_nested':training_nested(cache,cache['split']['train_ids'])}

def predict_boost(model,cache,doc_ids,batch_size,device,**kwargs):
    raw=raw_documents(model,cache,doc_ids,batch_size,device,floor=0.)
    return decode_raw(raw,cache['labels'],default_calibration(cache))

def blend_raw(all_raw,floor=-2.):
    if not all_raw:return {}
    if any(set(raw)!=set(all_raw[0]) for raw in all_raw):raise ValueError('Ensemble document IDs differ.')
    merged={};count=len(all_raw)
    for did in all_raw[0]:
        merged[did]={}
        for mode in ('learned','whitespace'):
            sums={}
            for raw in all_raw:
                for row in raw[did][mode]:
                    key=tuple(row[:3]);sums[key]=sums.get(key,floor*count)+(float(row[3])-floor)
            merged[did][mode]=[[*key,value/count] for key,value in sorted(sums.items())]
    return merged

def calibrated_decoder(raw,cache,grid=(0.,.5,-.5,1.,-1.,1.5,2.),min_support=5):
    from ner_data import strict_metrics
    labels=cache['labels'];gold={did:cache['gold'][did] for did in cache['split']['val_ids']}
    nested=training_nested(cache,cache['split']['train_ids']);best=None
    for mode in ('learned','whitespace'):
        for threshold in grid:
            cal={'boundary_mode':mode,'thresholds':dict.fromkeys(labels,threshold),'allow_nested':nested}
            pred=decode_raw(raw,labels,cal);metrics=strict_metrics(pred,gold,labels)
            quality=(metrics['macro_f1'],metrics['macro_precision'],-sum(len(e) for e in pred.values()))
            if best is None or quality>best[0]:best=(quality,cal,metrics)
    cal=best[1];counts={l:sum(e[2]==l for spans in gold.values() for e in spans) for l in labels}
    for label in labels:
        if counts[label]<min_support:continue
        one_gold={did:[e for e in spans if e[2]==label] for did,spans in gold.items()}
        best_quality=None;chosen=cal['thresholds'][label]
        for threshold in grid:
            one=dict(cal);one['thresholds']={**cal['thresholds'],label:threshold}
            pred=decode_raw(raw,[label],one)
            score=strict_metrics(pred,one_gold,[label])
            quality=(score['macro_f1'],score['macro_precision'],-sum(len(e) for e in pred.values()))
            if best_quality is None or quality>best_quality:best_quality=quality;chosen=threshold
        cal['thresholds'][label]=chosen
    predictions=decode_raw(raw,labels,cal)
    return cal,strict_metrics(predictions,gold,labels),predictions

def main():
    if len(sys.argv)>1 and sys.argv[1]=='train':
        sys.argv=[sys.argv[0],*sys.argv[2:]]
        import train_ner
        train_ner.build_model=build_boost;train_ner.NERWindows=SpanWindows
        train_ner.ner_loss=boost_loss;train_ner.predict_documents=predict_boost;train_ner.device_setup=setup_device
        train_ner.main();return
    p=argparse.ArgumentParser();p.add_argument('action',choices=['raw'])
    p.add_argument('--checkpoint',required=True);p.add_argument('--cache',required=True)
    p.add_argument('--split',choices=['val','test'],required=True);p.add_argument('--output',required=True)
    p.add_argument('--batch-size',type=int,default=2);args=p.parse_args()
    rank,world,device=setup_device();cache=torch.load(args.cache,map_location='cpu',weights_only=False)
    saved=torch.load(args.checkpoint,map_location='cpu',weights_only=False)
    if saved['cache_fingerprint']!=cache['fingerprint']:raise ValueError('Checkpoint/cache mismatch.')
    model,_=build_boost(saved['config'],cache['labels'],cache['char_classes'],cache['tokenizer_sha256'],args.checkpoint)
    model.to(device);root=Path(args.output);root.mkdir(parents=True,exist_ok=True)
    ids=cache['split']['val_ids'] if args.split=='val' else sorted(set(cache['texts'])-set(cache['gold']))
    identity=digest([file_hash(args.checkpoint),cache['fingerprint'],args.split,world,'raw-v1'])
    path=root/f'part_{rank}.json'
    saved_raw=json.loads(path.read_text()) if path.exists() else {'identity':identity,'raw':{}}
    if saved_raw['identity']!=identity:raise ValueError('Raw prediction identity changed; choose another output directory.')
    progress=saved_raw['raw']
    for did in ids[rank::world]:
        if did in progress:continue
        progress.update(raw_documents(model,cache,[did],args.batch_size,device))
        atomic_json(path,{'identity':identity,'raw':progress})
    if world>1:dist.barrier()
    if rank==0:
        combined={}
        for i in range(world):combined.update(json.loads((root/f'part_{i}.json').read_text())['raw'])
        if set(combined)!=set(ids):raise ValueError('Raw prediction IDs missing.')
        atomic_json(root/'raw.json',combined)
    if world>1:dist.barrier();dist.destroy_process_group()

if __name__=='__main__':main()
'''
(CODE / 'span_boost.py').write_text(BOOST_MODULE.lstrip('\n'), encoding='utf-8')

from ner_data import atomic_json, class_weights, collate_windows, digest, file_hash, read_json, strict_metrics, windows_for
from ner_predict import autocast, submission_zip
from phase2_workflow import launch
importlib.invalidate_caches()
import span_boost
span_boost = importlib.reload(span_boost)

base = torch.load(CACHE_PATH, map_location='cpu', weights_only=False)
if file_hash(foundation_root / 'tokenizer.json') != base['tokenizer_sha256']:
    raise ValueError('The attached Phase-1 tokenizer differs from the original NER cache.')
train_ids, val_ids = base['split']['train_ids'], base['split']['val_ids']
labels = base['labels']
counts = lambda ids: {label:sum(e[2] == label for did in ids for e in base['gold'][did]) for label in labels}
train_counts, val_counts = counts(train_ids), counts(val_ids)
ceiling = sum(value > 0 for value in val_counts.values()) / len(labels)
print('Validation label support:', val_counts)
print('Maximum possible strict macro-F1 on this split:', round(ceiling, 4))
if ceiling < 0.75:
    print('TARGET LIMIT: 0.75 is impossible on this holdout because some labels have no gold examples. The official metric still includes all 12 labels.')

# Compare exact-span and token-boundary scores without changing the official metric.
catalog_path = PROJECT / 'catalog.json'
baseline = None
if catalog_path.exists():
    catalog = read_json(catalog_path)
    if catalog:
        baseline = max(catalog, key=lambda item:item['validation_macro_f1'])
        print('Best existing held-out strict macro-F1:', baseline['name'], baseline['validation_macro_f1'])
        prediction_path = PROJECT / 'runs' / baseline['name'] / 'selected_validation/prediction.json'
        if prediction_path.exists():
            prediction = read_json(prediction_path)
            def token_boundaries(rows):
                result = {}
                for did, spans in rows.items():
                    offsets = np.asarray(base['encoded'][did]['offsets']).reshape(-1, 2)
                    entities = set()
                    for a, b, label, *_ in spans:
                        hits = np.flatnonzero((offsets[:, 1] > a) & (offsets[:, 0] < b))
                        if len(hits): entities.add((int(hits[0]), int(hits[-1]), label))
                    result[did] = [list(entity) for entity in entities]
                return result
            gold = {did:base['gold'][did] for did in val_ids}
            print('Diagnostic token-boundary macro-F1 (NOT the official score):',
                  strict_metrics(token_boundaries(prediction), token_boundaries(gold), labels)['macro_f1'])

context = max(BOOST_CONTEXT, base['seq_len'])
overlap = max(256, base['overlap'])
if context > 4096 or overlap >= context:
    raise ValueError('Adjust BOOST_CONTEXT: overlap must be smaller than context and context must fit Phase-1 positions.')
boost_cache = dict(base)
boost_cache['seq_len'], boost_cache['overlap'] = context, overlap

def training_windows(ids):
    regular = windows_for(base['encoded'], ids, context, overlap)
    limit = math.ceil(len(regular)*1.5)
    seen = {(w['did'], w['start'], w['end']) for w in regular}
    support = counts(ids)
    extra = []
    for did in ids:
        n = len(base['encoded'][did]['ids'])
        for i, j, label, *_ in base['aligned'][did]['entities']:
            start = max(0, min((i+j)//2-context//2, max(0, n-context)))
            end = min(n, start+context)
            key = (did, start, end)
            if start <= i <= j < end and key not in seen:
                extra.append((support[labels[label]], digest(key), {'did':did, 'start':start, 'end':end}))
    # Add at most 50% more windows, prioritizing contexts containing scarce labels.
    for _, _, window in sorted(extra, key=lambda row:(row[0], row[1])):
        key = (window['did'], window['start'], window['end'])
        if key in seen: continue
        regular.append(window);seen.add(key)
        if len(regular) >= limit: break
    return regular

boost_cache['train_windows'] = training_windows(train_ids)
boost_cache['all_windows'] = training_windows(sorted(base['gold']))
boost_cache['val_windows'] = windows_for(base['encoded'], val_ids, context, overlap)
boost_cache['test_windows'] = windows_for(base['encoded'], sorted(set(base['texts'])-set(base['gold'])), context, overlap)
boost_cache['fingerprint'] = digest({'parent_cache':base['fingerprint'], 'extension':'bidir-span-v1',
    'context':context, 'overlap':overlap, 'train_windows':boost_cache['train_windows'], 'all_windows':boost_cache['all_windows']})
assert boost_cache['split'] == base['split']
BOOST_CACHE_PATH = BOOST_ROOT / 'cache.pt'
if BOOST_CACHE_PATH.exists():
    saved_cache = torch.load(BOOST_CACHE_PATH, map_location='cpu', weights_only=False)
    if saved_cache['fingerprint'] != boost_cache['fingerprint']:
        raise ValueError('Extension cache changed. Choose a new BOOST_ROOT_NAME.')
    del saved_cache
else:
    temporary = BOOST_CACHE_PATH.with_suffix('.tmp.pt')
    torch.save(boost_cache, temporary);temporary.replace(BOOST_CACHE_PATH)
print('Additional-model windows:', {s:len(boost_cache[s+'_windows']) for s in ('train', 'val', 'all', 'test')})

base_config = {'architecture':'decoder', 'method':'full', 'head':'span',
    'phase1_checkpoint':str(foundation_checkpoint), 'two_directions':False, 'span_dim':64,
    'max_span_tokens':context, 'candidate_cap':4096, 'head_dropout':.20, 'backbone_dropout':.10,
    'boundary_weight':.75, 'aux_bio_weight':.30, 'backbone_lr':2e-5, 'head_lr':2e-4,
    'weight_decay':.05, 'grad_clip':1., 'epochs':BOOST_EPOCHS, 'patience':BOOST_PATIENCE,
    'micro_batch':2, 'accumulation':4, 'inference_batch':2, 'workers':0,
    'max_stage_hours':BOOST_MAX_STAGE_HOURS, 'gradient_checkpointing':False,
    'checkpoint_every':50, 'log_every':20, 'seed':BOOST_SEEDS[0], 'allow_tiny':False}

# Measure a real full-length training micro-batch before launching both GPUs.
device = torch.device('cuda', 0)
while True:
    model = optimizer = output = loss = batch = weights = scaler = None
    try:
        torch.cuda.empty_cache()
        model, _ = span_boost.build_boost(base_config, labels, boost_cache['char_classes'], boost_cache['tokenizer_sha256'])
        model.to(device).train()
        dataset = span_boost.SpanWindows(boost_cache, 'train', 'span')
        longest = max(range(len(dataset)), key=lambda index:len(dataset[index]['ids']))
        batch = collate_windows([dataset[longest] for _ in range(base_config['micro_batch'])])
        for key in ('ids', 'valid', 'tags', 'left', 'right'): batch[key] = batch[key].to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, fused=True)
        weights = class_weights(boost_cache, 'train').to(device)
        scaler = torch.amp.GradScaler('cuda')
        torch.cuda.reset_peak_memory_stats(device)
        for attempt in range(10):
            optimizer.zero_grad(set_to_none=True)
            with autocast(device):
                output = model(batch['ids'], batch['valid'])
                loss = span_boost.boost_loss(output, batch, base_config, weights)
            if not bool(torch.isfinite(loss)): raise FloatingPointError('Additional-model preflight loss is nonfinite.')
            scaler.scale(loss).backward();scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.)
            old_scale = scaler.get_scale();scaler.step(optimizer);scaler.update()
            if scaler.get_scale() >= old_scale: break
        else: raise FloatingPointError('Preflight could not complete a finite FP16 optimizer update.')
        peak = torch.cuda.max_memory_reserved(device)/1024**3
        if peak > 12.5: raise torch.cuda.OutOfMemoryError('Reserve memory for DDP.')
        print('Additional-model preflight passed:', round(peak, 2), 'GiB; micro_batch', base_config['micro_batch'])
        break
    except torch.cuda.OutOfMemoryError:
        if base_config['micro_batch'] > 1:
            base_config['micro_batch'] = 1;base_config['accumulation'] = 8;base_config['inference_batch'] = 1
        elif not base_config['gradient_checkpointing']:
            base_config['gradient_checkpointing'] = True
        else: raise RuntimeError('Reduce BOOST_CONTEXT and choose a new BOOST_ROOT_NAME.')
    finally:
        del model, optimizer, output, loss, batch, weights, scaler
        gc.collect();torch.cuda.empty_cache()

configs = []
validation_raw = []
for index, seed in enumerate(BOOST_SEEDS):
    run_root = BOOST_ROOT / f'seed_{seed}'
    run_root.mkdir(exist_ok=True)
    cfg = {**base_config, 'name':f'08_bidir_span_seed_{seed}', 'seed':seed}
    cfg_path = run_root / 'config.json'
    if cfg_path.exists() and read_json(cfg_path) != cfg:
        raise ValueError('Training settings changed; choose a new BOOST_ROOT_NAME.')
    atomic_json(cfg_path, cfg);configs.append((cfg, cfg_path, run_root))
    launch('span_boost.py', ['train', '--config', cfg_path, '--cache', BOOST_CACHE_PATH,
        '--output', run_root/'selection', '--split', 'train'], PROJECT, 2, run_root/'selection.log')
    launch('span_boost.py', ['raw', '--checkpoint', run_root/'selection/best.pt', '--cache', BOOST_CACHE_PATH,
        '--split', 'val', '--output', run_root/'validation_raw', '--batch-size', cfg['inference_batch']], PROJECT, 2, run_root/'validation.log')
    validation_raw.append(read_json(run_root/'validation_raw/raw.json'))

variants = [(f'seed_{seed}', raw, [index]) for index, (seed, raw) in enumerate(zip(BOOST_SEEDS, validation_raw))]
if len(validation_raw) > 1:
    variants.append(('mean_ensemble', span_boost.blend_raw(validation_raw), list(range(len(configs)))))
choices = []
for name, raw, members in variants:
    calibration, metrics, predictions = span_boost.calibrated_decoder(raw, boost_cache)
    choices.append((name, calibration, metrics, predictions, members))
    print(name, 'validation-tuned strict macro-F1:', round(metrics['macro_f1'], 5))
best_name, calibration, metrics, validation_prediction, members = max(choices, key=lambda item:item[2]['macro_f1'])
atomic_json(BOOST_ROOT/'calibration.json', {'variant':best_name, 'members':members, **calibration})
atomic_json(BOOST_ROOT/'validation_metrics.json', metrics)
atomic_json(BOOST_ROOT/'validation_prediction.json', validation_prediction)
print('Selected additional candidate:', best_name)
print('Label              train  validation   strict-F1')
for row in metrics['per_label']:
    print(f"{row['label']:<18} {train_counts[row['label']]:>6} {val_counts[row['label']]:>10} {row['f1']:>11.4f}")

test_raw = []
for member in members:
    cfg, cfg_path, run_root = configs[member]
    selected_checkpoint = run_root/'selection/best.pt'
    if BOOST_REFIT:
        best_epoch = max(1, read_json(run_root/'selection/summary.json')['best_epoch'])
        launch('span_boost.py', ['train', '--config', cfg_path, '--cache', BOOST_CACHE_PATH,
            '--output', run_root/'refit', '--split', 'all', '--epochs', best_epoch], PROJECT, 2, run_root/'refit.log')
        selected_checkpoint = run_root/'refit/best.pt'
    launch('span_boost.py', ['raw', '--checkpoint', selected_checkpoint, '--cache', BOOST_CACHE_PATH,
        '--split', 'test', '--output', run_root/('test_refit_raw' if BOOST_REFIT else 'test_selection_raw'),
        '--batch-size', cfg['inference_batch']], PROJECT, 2, run_root/'prediction.log')
    test_raw.append(read_json(run_root/('test_refit_raw' if BOOST_REFIT else 'test_selection_raw')/'raw.json'))
merged_test = span_boost.blend_raw(test_raw) if len(test_raw) > 1 else test_raw[0]
prediction = span_boost.decode_raw(merged_test, labels, calibration)
texts = {did:boost_cache['texts'][did] for did in prediction}
submission_path = PROJECT/'submissions/08_bidir_span_boost.zip'
prediction_hash = submission_zip(prediction, texts, labels, submission_path)
atomic_json(BOOST_ROOT/'prediction.json', prediction)
atomic_json(BOOST_ROOT/'report.json', {'candidate':best_name, 'validation_tuned_macro_f1':metrics['macro_f1'],
    'validation_macro_f1_ceiling':ceiling, 'original_best_macro_f1':baseline['validation_macro_f1'] if baseline else None,
    'split_fingerprint':digest(base['split']), 'validation_reused_for_calibration':True,
    'refit_on_all_labels':BOOST_REFIT, 'score_is_not_private_test_f1':True,
    'calibration':calibration, 'submission':str(submission_path), 'prediction_sha256':prediction_hash,
    'candidate_floor':-2., 'candidate_cap_per_window':4096})
print('Additional submission ready:', submission_path)
print('Measured validation-tuned strict macro-F1:', round(metrics['macro_f1'], 5))
print('This score used validation for checkpoint/model/threshold selection; it is not an independent test estimate.')
if baseline and metrics['macro_f1'] <= baseline['validation_macro_f1']:
    print('The additional candidate did not improve the existing held-out result. Keep your original submissions.')
if metrics['macro_f1'] < 0.75:
    print('The requested 0.75 target was not reached on this holdout. Inspect validation_metrics.json for the limiting labels.')
display(FileLink(str(submission_path)))
display(FileLink(str(BOOST_ROOT/'validation_metrics.json')))


## Compare quality and efficiency; inspect each label

Use the same document holdout for all seven. Compare strict F1, trainable/total parameters and observed training time together. A cheaper method is useful only if its quality is acceptable. If all-label refit hit its own cap, it may receive fewer updates than intended; its summary records this. Training-stage caps do not establish equal compute or fair convergence.

Inspect the per-label table and boundary examples before uploading. BIO cannot represent overlapping entities; the audit reports omissions rather than hiding them. Identical prediction hashes are explicitly flagged and do not establish distinct accepted submissions.

In [ ]:
import pandas as pd
catalog = read_json(PROJECT / 'catalog.json')
summary_table = pd.DataFrame([{
    'candidate': c['name'], 'foundation': c['architecture'], 'method': c['method'],
    'strict_macro_f1': c['validation_macro_f1'], 'selected_epoch': c['selection_epoch'],
    'total_parameters': c['parameters']['total_parameters'],
    'trainable_parameters': c['parameters']['trainable_parameters'],
    'selection_minutes': c['selection_seconds'] / 60, 'test_entities': c['entities'],
} for c in catalog]).sort_values('strict_macro_f1', ascending=False)
display(summary_table)
per_label = pd.DataFrame([{'candidate': c['name'], **row} for c in catalog for row in c['metrics']['per_label']])
display(per_label.pivot(index='label', columns='candidate', values='f1'))
per_label.to_csv(PROJECT / 'per_label_metrics.csv', index=False)
best = max(catalog, key=lambda c: c['validation_macro_f1'])
print('Best measured held-out candidate:', best['name'], 'Macro-F1:', best['validation_macro_f1'])
print('This is a held-out score, not a private leaderboard score.')
best_per_foundation = [max((c for c in catalog if c['architecture'] == architecture),
                          key=lambda c: c['validation_macro_f1']) for architecture in foundations]
atomic_json(PROJECT / 'best_per_foundation.json', [{k: c[k] for k in ['name', 'architecture', 'validation_macro_f1', 'submission']}
                                                for c in best_per_foundation])
print('For the required four-foundation comparison:', [c['name'] for c in best_per_foundation])

In [ ]:
best_predictions = read_json(PROJECT / 'runs' / best['name'] / 'selected_validation' / 'prediction.json')
errors = []
for did in cache['split']['val_ids']:
    reference = {tuple(e[:3]) for e in gold[did]}; predicted = {tuple(e[:3]) for e in best_predictions[did]}
    for category, spans in [('false_positive', predicted - reference), ('false_negative', reference - predicted)]:
        for start, end, label in sorted(spans):
            errors.append({'document_id': did, 'category': category, 'label': label, 'start': start, 'end': end,
                           'entity_text': train_texts[did][start:end],
                           'context': train_texts[did][max(0, start - 70): min(len(train_texts[did]), end + 70)]})
atomic_json(PROJECT / 'best_validation_errors.json', errors)
display(pd.DataFrame(errors[:30]))

## Verify and download seven individual submission ZIPs

Each ZIP has only `prediction.json` at the root. IDs exactly match the official test set; offsets reference the untouched original document with an exclusive end; labels are from `labels.txt`; no duplicate spans are emitted. No test labels are used. The bundle below is for downloading all seven files together: **extract it and upload the individual ZIPs**, not the bundle itself.

In [ ]:
from IPython.display import FileLink, display
import zipfile
hashes = []
for candidate in catalog:
    path = Path(candidate['submission'])
    with zipfile.ZipFile(path) as z:
        assert z.namelist() == ['prediction.json']
        prediction = json.loads(z.read('prediction.json'))
    validate_spans(prediction, test_texts, labels)
    hashes.append(digest(prediction))
    display(FileLink(str(path.relative_to(WORKING))))
assert len(catalog) == 7
print('Valid ZIPs:', len(catalog), 'Unique prediction contents:', len(set(hashes)))
if len(set(hashes)) != 7:
    print('WARNING: Some independently trained methods produced identical predictions. Improve the affected model; do not submit renamed duplicates as distinct predictions.')
download_bundle = WORKING / 'phase2_seven_submissions.zip'
with zipfile.ZipFile(download_bundle, 'w', zipfile.ZIP_DEFLATED) as z:
    for candidate in catalog: z.write(candidate['submission'], Path(candidate['submission']).name)
    z.write(PROJECT / 'comparison.csv', 'comparison.csv')
    z.write(PROJECT / 'catalog.json', 'catalog.json')
display(FileLink(download_bundle.name))

## Provenance and assistance declaration draft

This records inputs, methods and assistance for your report. Review it against the current course policy and any professor clarification. The reviewed capstone contains assistant-use restrictions; this draft does not assert permission beyond those instructions.

In [ ]:
declaration = {
    'course': 'CS781', 'phase': 2, 'student_roll': '230437',
    'foundation_source': 'Student own Phase-1 checkpoints; no external pretrained model weights',
    'tokenizer_sha256': cache['tokenizer_sha256'], 'dataset_fingerprint': cache['fingerprint'],
    'assistance': 'Notebook code and research synthesis prepared with ChatGPT/Codex assistance',
    'permission_context': 'Review assistance against current course instructions and any professor clarification',
    'methods': [{k: row[k] for k in ['name', 'architecture', 'method', 'head']} for row in rows],
    'evaluation': 'Document-level holdout; exact span strict Macro-F1 over all 12 labels',
    'refit_all_labelled_data': SETTINGS['refit_all'],
    'private_test_labels_used': False,
    'submitted_to_platform_by_notebook': False,
}
atomic_json(PROJECT / 'assistance_and_provenance_draft.json', declaration)
print('Saved declaration draft, source code, split, provenance, metrics and run logs.')

## Complete `/kaggle/working` archive — run this cell last

This includes the seven candidate ZIPs, restored foundations, adapted best checkpoints, recovery states, code, cached data/teacher logits, metrics and logs. It excludes itself, its temporary ZIP and Python bytecode caches. It checks free space before copying and validates ZIP CRCs. If disk space is insufficient, the seven small submission files and their download bundle already exist; download those first. A full archive can be several GB.

Wait for the ZIP cell to finish, **Save Version → Save & Run All**, and download the archive from that version's Output tab after completion. Kaggle's CLI `kernels output` downloads saved output files; it cannot recover unsaved files from a stopped interactive session.

In [ ]:
# ALL_OUTPUTS_ZIP = WORKING / 'phase2_all_kaggle_working.zip'
# archive_path = bundle_working(WORKING, ALL_OUTPUTS_ZIP)
# print('Complete output archive:', archive_path, 'size GB:', round(Path(archive_path).stat().st_size / 1e9, 3))
# display(FileLink(ALL_OUTPUTS_ZIP.name))

# CS781 Phase 2: seven experiments on two T4 GPUs

Prepared against the original [Capstone Document](https://docs.google.com/document/d/1olqoT5KPhf5CSJLBNjXGyKyyyQDBr6jcnWqPUh5cNk4/edit), updated October 9, 2026, including Important Instructions, Phase 2, FAQs and the linked official Phase-2 evaluator. The document states a Phase-2 deadline of **October 11, 2026, 11:59 PM**; the course website is the authority for any subsequent extension.

## Required inputs and outputs

Attach your extracted `capstone_phase1` directory as a Kaggle dataset. It must contain the exact `tokenizer.json` and four `runs/main_<architecture>/best_model.pt` checkpoints: `decoder`, `encoder_decoder`, `local_gqa`, and `hybrid`. Keep that folder unchanged. Its `benchmarks/foundation_<architecture>/` directories contain benchmark configurations and timings. The loader uses only the five configured files from the direct `capstone_phase1` folder. It performs no recursive discovery or ZIP extraction, so duplicate copies elsewhere in the input dataset cannot cause the multiple-output error. Phase-1 pretraining is not repeated.

The delivered notebook is configured explicitly with your supplied root `/kaggle/input/datasets/hardik4tiwari/phase-1-outputs/capstone_phase1`, its four `runs/main_<architecture>/best_model.pt` files and its root `tokenizer.json`. No path edits are needed for that attachment. The `all_kaggle_outputs/capstone_phase1` copy is excluded. Setting `PHASE1_ROOT = None` uses the same direct root rather than enabling discovery. If you move the dataset, update the root, checkpoint and tokenizer paths together. The loader cell reloads the rewritten module so an existing Kaggle session uses the current implementation.

Configured full output attachment (other existing files can remain):

```text
/kaggle/input/datasets/hardik4tiwari/phase-1-outputs/capstone_phase1/
    tokenizer.json
    foundation_manifest.json
    benchmarks/
        foundation_decoder/
        foundation_encoder_decoder/
        foundation_local_gqa/
        foundation_hybrid/
    cache/
    runs/
        main_decoder/best_model.pt
        main_encoder_decoder/best_model.pt
        main_local_gqa/best_model.pt
        main_hybrid/best_model.pt
    submissions/
    submission_candidates/
```

Attach the official six Phase-2 dataset files, or accept the terms of `Exploration-Lab/CS-781-Capstone` and create a Kaggle secret named `HF_TOKEN`. Enable Kaggle Internet for installation/downloads. Never paste access tokens into notebook cells. The dataset revision is pinned and its file hashes recorded. The gated annotations were unavailable in the development environment, so the notebook checks their actual schema and offset representability when run.

Select **GPU T4 ×2**. The notebook saves everything it creates under `/kaggle/working/capstone_phase2`. It creates seven platform-ready ZIPs, each containing **only `prediction.json` at its root**, plus an experiment comparison and a final complete working-directory archive. Upload the seven individual ZIPs to the course site; generating them locally is not seven accepted leaderboard submissions. Verify the current daily limit before uploading; the reviewed document states ten per day.

## Experiment matrix

| Candidate | Phase-1 foundation | Adaptation | NER head |
|---|---|---|---|
| 01 | Causal decoder | Full fine-tuning | BIO + learned character trims |
| 02 | Encoder–decoder | Full fine-tuning | BIO + learned character trims |
| 03 | Local-attention GQA | LoRA | BIO + learned character trims |
| 04 | LSTM–transformer hybrid | Last three blocks and final norm | BIO + learned character trims |
| 05 | Causal decoder | LoRMA+ | GlobalPointer spans + character trims |
| 06 | Causal decoder | Residual bottleneck adapters | BIO + learned character trims |
| 07 | Compressed decoder student | Supervised learning + teacher-logit distillation | BIO + learned character trims |

The four foundations are reused at their original dimensions, around 50 million parameters each. The student retains four uniformly spaced blocks from the original eight-block decoder and copies the embedding and final normalization, making its backbone approximately 33.6 million parameters. Actual counts, including task heads, are recorded. The teacher is the best validation BIO experiment and must have more total parameters than the student. The span-head experiment is excluded from BIO-logit distillation because its output space differs.

The reviewed document requires all four foundations, several adaptation techniques, compulsory LoRMA and a smaller distilled student. Its updated Important Instructions require at least six submissions in Phases 2–7 although the Phase-2 section still says four. This seven-experiment design covers the stricter count. It does not establish that a particular implementation will be accepted as a technique by the course staff; descriptions accurately name LoRMA+ and the adapter variant.

## Research: methods, implementation choices and limits

### LoRMA: Bihany, Patel and Modi, Findings of ACL 2025

[Paper, including appendices](https://aclanthology.org/2025.findings-acl.527.pdf) · [Authors' repository](https://github.com/Exploration-Lab/LoRMA)

The method and complexity appendix distinguish multiplicative adaptation from additive LoRA. We implement **LoRMA+**, Equation 8:

\[
y=W_0x,\qquad h=y+\frac{\alpha}{r}B(Ay).
\]

Here `A` maps output width to rank and `B` maps rank back to output width. Random `A` and zero `B` preserve the foundation at initialization. Training freezes `W0` and computes the low-rank residual after its ordinary projection. It never constructs a dense output-by-output transform. This choice avoids the dense transformed-matrix work discussed for the permutation variant. The paper reports convergence and parameter savings on its own tasks, not legal NER on T4s. Our validation, timing and trainable-parameter logs provide the relevant measurements. We adapt attention and feed-forward projections, and add a separately trained span head; that combination is our experiment rather than a reported paper result.

### LoRA: Hu et al., 2021

[Paper](https://arxiv.org/pdf/2106.09685)

For additive adaptation, the notebook implements `W0 x + (alpha/r) B A x`, with `A` mapping **input** width to rank, unlike LoRMA+. Its initialization follows the paper's random/zero factors. Rank 8 and alpha 16 are starting choices, not claimed optima. Freezing the backbone reduces optimizer-state and gradient storage, but later adapters still require gradients through earlier operations. Consequently, a large reduction in trainable parameters does not imply an equally large reduction in training time. A unit check compares each implementation with its mathematically merged weight after a nonzero update, which also checks rectangular feed-forward projections. The notebook retains adapters explicitly in task checkpoints; it does not claim that they were merged for inference.

### Adapters: Houlsby et al., ICML 2019

[Paper](https://arxiv.org/pdf/1902.00751)

The paper motivates task-specific residual bottlenecks and updating a small fraction of foundation parameters. The implementation adds `up(GELU(down(norm(h))))` after each transformer block, with a zero-initialized up projection. It also trains selected normalization parameters. This is a **post-block bottleneck variant**, not an exact reproduction of the paper's two insertions around attention and feed-forward sublayers. Bottleneck width 64 gives a distinct adaptation route from low-rank projection changes. It preserves the initial block output while letting the task learn new residual features. Backbone optimizer memory falls, but sequential bottleneck computation still contributes inference and training cost; actual measurements decide its usefulness.

### Distillation: Hinton, Vinyals and Dean, 2015

[Paper](https://arxiv.org/pdf/1503.02531)

The student minimizes a weighted combination of hard-label cross-entropy and softened teacher–student KL divergence. Temperature scaling uses the paper's `T²` correction; defaults are temperature 2 and soft-loss weight 0.5. Boundary heads still learn directly from official annotations. Teacher logits are cached in FP16 for precisely the training windows used by the student, reducing repeated teacher forward passes. Cache identities include teacher-checkpoint hash, dataset fingerprint, split and window geometry. Validation-stage logits come from a teacher trained on the training partition only. If all-label refitting is enabled, a separate teacher refit and cache are used; those weights do not supply validation claims. The smaller student's quality and cost are compared empirically, with no promised retention percentage.

### Legal NER: IL-TUR, 2024

[Paper, including task and dataset appendices](https://arxiv.org/pdf/2407.05399)

Its legal entity task and analysis motivate distinguishing legal roles and preserving precise span boundaries. Our notebook addresses role context by sharing the original causal foundation across forward and reversed windows, then concatenating representations. The encoder–decoder instead obtains bidirectional context from its encoder using the same unmodified input text; no labels enter the encoder. These choices change task computation while retaining the user's original foundation weights. They cost additional compute and are not advertised as free speedups. Character-trim heads address BPE tokens containing leading whitespace or multiple characters. No benchmark model, external pretrained tokenizer or reported benchmark score is imported into this project.

### GlobalPointer: Su et al., 2022

[Paper](https://arxiv.org/pdf/2208.03054)

Candidate 05 assigns each entity label token-start and token-end vectors, applies rotary positions and scores their dot products. The triangular mask permits end positions at or after the start; padding is excluded. Training uses the paper's positive/negative log-sum-exp objective from Equation 13, with zero as the reference logit; decoding initially uses threshold zero. It can represent nested or overlapping entities that a single BIO sequence cannot. A separate learned character-trim head converts token boundaries back to precise source offsets. Its score tensor grows as labels times sequence length squared, so default windows are 512 tokens and GPU memory is measured first. This is the original span formulation, not an Efficient GlobalPointer implementation. The recorded per-window candidate cap is a computational guard and can affect recall.

### Structured decoding: Ma and Hovy, ACL 2016

[Paper](https://aclanthology.org/P16-1101.pdf)

The paper combines contextual token representations with a trained CRF and Viterbi decoding. Here we borrow the need for sequence-consistent decoding but use **fixed BIO legality constraints**: an I-tag cannot start a document or follow an unrelated label. The emissions are trained with weighted token cross-entropy; there is no learned transition matrix or CRF likelihood. Decode after merging all overlapping-window emissions for the document, avoiding artificial chunk-boundary entities. This preserves a straightforward objective and prevents invalid label transitions, while leaving the question of a learned CRF's benefit for a subsequent measured experiment.

### Mixed precision: Micikevicius et al., 2017

[Paper](https://arxiv.org/pdf/1710.03740)

T4s support FP16; the notebook uses autocast and dynamic gradient scaling while keeping parameters and optimizer state in FP32. Losses, normalization reductions and log-sum-exp/KL computations use FP32 where needed. Gradients are unscaled before clipping; skipped scaled updates are logged and do not advance the successful-update schedule. The two GPUs use data parallelism, each holding a complete model replica; they do not form one 30 GB memory pool. Accumulation increases the effective batch without raising each micro-batch's activation footprint. It is not a substitute for measuring memory or numerical stability. Actual T4 execution remains necessary to validate the CPU-tested code's performance.

### Attention efficiency: Dao et al., NeurIPS 2022

[FlashAttention paper](https://arxiv.org/pdf/2205.14135) · [Official implementation and hardware support](https://github.com/Dao-AILab/flash-attention)

The paper explains reducing device-memory traffic through tiled exact attention rather than approximating attention scores. The notebook uses PyTorch scaled-dot-product attention, which chooses a supported backend for the current GPU, dtype and mask. It does **not** assume FlashAttention-2 support on Turing T4 hardware or install an incompatible extension. Padding and local-attention masks may change dispatch and runtime. The inherited local-GQA model uses dense masked attention; this is not a sparse-attention speed claim. Benchmark every architecture instead of extrapolating from the paper's different hardware, shapes and kernels.

### Activation checkpointing: Chen et al., 2016

[Paper](https://arxiv.org/pdf/1604.06174)

Checkpointing reduces stored activations by recomputing intermediate forward operations during backward propagation. It trades additional computation for memory. The notebook supports non-reentrant block checkpointing, but leaves it disabled initially because these roughly 50M foundations may fit the target window and micro-batch. First measure the largest-window training step with optimizer state allocated, leave headroom for distributed communication, and reduce the micro-batch if necessary. Enable checkpointing if micro-batch one still does not fit, then rebuild the run configuration. It can make an otherwise impossible run feasible; it is not described as an unconditional training-speed improvement.

## Validation, runtime and reproducibility

The split groups whitespace/case duplicates and keeps complete documents together. A deterministic label-presence heuristic attempts coverage across all twelve labels, and the notebook prints both partitions' coverage. A label present in too few document groups cannot be reliably evaluated by a single split; interpret its reported F1 accordingly. BIO overlap omissions are counted explicitly, while the span head retains overlapping annotations. Every gold entity must be representable and fit in at least one training window.

Selection uses exact `(start, end, label)` matches and averages over **all twelve** labels, including zero-F1 absent labels, as in the official evaluator. A held-out score is never a private-test score. Each candidate optionally refits from the same original foundation on all labelled documents for its selected number of epochs. Its displayed score remains the earlier honest held-out score. Teacher selection also remains based on that held-out measurement.

Defaults allow twelve selection epochs with patience three, micro-batch four per GPU, accumulation four, 512-token windows and 128-token overlap. Overlap increases when necessary to contain training spans. **The 0.30-hour cap applies separately to each selection and refit training invocation.** Fourteen capped stages permit about 4.2 hours inside their training loops; validation at the end of a stage, teacher caching, inference, setup and archiving add time. The measured preflight forecast is approximate and excludes several overheads. Reduce budgets or disable refit if the observed forecast would jeopardize submission time. Allow time for uploads and the course's stated 15–30 minute scoring delay.

Every run stores configuration, provenance, parameter counts, per-label metrics, timings, best task weights and recovery states. Resume requires unchanged configuration, data, teacher and GPU count. An interrupted accumulation group resumes from the last saved optimizer step. Completed candidates remain immediately available. The final archive excludes itself and its temporary file and checks available disk space and ZIP integrity.

Seven distinct experiments normally produce seven different predictions, but identical outputs are detected and reported. The notebook never edits predictions randomly or renames duplicate results to claim distinct submissions. It records actual output hashes and entity counts so collapsed models are visible.

## Development validation and remaining checks

The deliverable is validated with synthetic own-foundation checkpoints, real ByteLevel BPE offsets, Unicode and whitespace boundaries, all seven losses and gradients, frozen-parameter checks, padding invariance, low-rank algebra, strict-metric parity with the downloaded official evaluator, and end-to-end single-process CPU training and inference. Interrupting and resuming a CPU run reproduced uninterrupted model and optimizer tensors exactly. Teacher-cache split checks, direct-folder loading in the presence of duplicate outputs, and complete-output archive integrity also passed. Two-process CPU/Gloo testing was blocked by this environment's network-interface permissions. The notebook therefore includes an actual two-GPU smoke test for every method before long training. These tests verify implementation behavior, not trained accuracy. GPU timings and official-data annotation audits must be checked on Kaggle; neither real Phase-1 weights nor gated annotations were available to the development environment.

The notebook creates an assistance/provenance declaration draft recording that code was prepared with ChatGPT/Codex assistance. Review it against current course instructions and any professor clarification before including it in a report. It does not assert permission beyond what those instructions authorize.